# 🏁 F1 Podium Prediction — Milestone 1
## Part 1 · Setup → Raw audit (EDA) → Cleaning → Re-EDA

**Task:** predict whether a driver finishes on the podium (`positionOrder <= 3`) using only information available **after qualifying and before the race**.
**Unit of analysis:** one driver entering one race → `(raceId, driverId)` must be unique.

| § | Section | Brief requirement it covers |
|---|---|---|
| 0 | Setup & config | run-all ready, Kaggle/local paths, seeds |
| 1 | Load raw data (immutable) | raw CSVs never modified, hashes checked at the end |
| 2 | **Raw audit — EDA before cleaning** | sentinels, PK/FK checks, duplicate grain, target, qualifying coverage, leakage map |
| 3 | **Cleaning** | `\N` sentinels, semantic typing, one row per driver-race, non-starters |
| 4 | **Re-EDA — EDA after cleaning** | before/after impact of every cleaning step |
| 5 | Decision log & hand-off | what Part 2 (questions, features, models, XAI) will consume |
| 6 | **Data-engineering Q1** (Part 2 starts here) | ranking of circuits, grid vs qualifying |
| 7 | **Data-engineering Q2** | home race effect, controlling for grid |
| 8 | **Data-engineering Q3** | mechanical-retirement rate per constructor |
| 9 | **Feature selection & split** | training years, leakage-safe features, correlation maps, importance, chronological split |
| 10 | **Training** | baseline + 2 statistical models + shallow neural network, tuning, thresholds, metrics |
| 11 | **Experiments** | feature-group ablation, preprocessing order |
| 12 | **Test** (2022–24, run once) | all four models, expected vs predicted |
| 13 | **Explainability** | permutation importance, SHAP, LIME / local SHAP |
| 14 | **Inference function** | complete and partly-missing record |
| 15 | **Limitations & references** | era sensitivity, qualifying drift, non-causal reading |

> **Running on Kaggle:** *Add Data* → search **“Formula 1 World Championship (1950-2020)”** (by Rohan Rao) → add it → *Run All*. §13 installs `shap` and `lime` if they are missing, so switch **Internet on** in the notebook settings.
> Everything below is deterministic; nothing is edited by hand between cells.

## 0 · Setup & configuration
The two `CFG` switches are the only judgement calls in this part. They are exposed so the team can flip them and see the effect (the notebook reports the effect of each).

In [ ]:
import re, time, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 170)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

SEED = 42
np.random.seed(SEED)

# ── judgement calls (change here, re-run, compare) ───────────────────────────
CFG = dict(
    dup_rule='best_position',       # 'best_position' | 'most_laps' | 'first_result'  (see §3.3)
    exclude_non_starters=True,      # drop entries that never got a grid slot (DNQ/DNPQ/withdrew) (see §3.4)
)

# ── paths: Kaggle input OR local data/raw ────────────────────────────────────
ON_KAGGLE = Path('/kaggle/input').exists()

def find_raw_dir():
    roots = ([Path('/kaggle/input')] if ON_KAGGLE else []) + [Path('data/raw'), Path('../data/raw')]
    for root in roots:
        if root.exists():
            for p in sorted(root.rglob('results.csv')):
                return p.parent
    raise FileNotFoundError('results.csv not found - add the Kaggle F1 dataset or put the CSVs in data/raw/')

RAW_DIR  = find_raw_dir()
OUT_DIR  = Path('/kaggle/working') if ON_KAGGLE else RAW_DIR.parent.parent / 'outputs'
FIG_DIR  = OUT_DIR / 'figures'
PROC_DIR = OUT_DIR / 'processed'
for d in (FIG_DIR, PROC_DIR): d.mkdir(parents=True, exist_ok=True)
print('raw data :', RAW_DIR.resolve())
print('outputs  :', OUT_DIR.resolve())
print('kaggle   :', ON_KAGGLE)

# ── plotting style (one colour per job: blue = raw / before, orange = clean / after) ──
BLUE, ORANGE, AQUA, MAGENTA = '#2a78d6', '#eb6834', '#1baf7a', '#e87ba4'
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e6e5e1'
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white',
    'axes.edgecolor': GRID, 'axes.labelcolor': INK2, 'xtick.color': INK2, 'ytick.color': INK2,
    'text.color': INK, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'font.size': 10, 'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'legend.frameon': False, 'figure.dpi': 100,
})

def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'{name}.png', dpi=130, bbox_inches='tight')
    plt.show()

## 1 · Load raw data (immutable)
**Why this way (justification):**
* Every CSV is read straight from the input folder and **never written back**. We store an MD5 hash per file now and re-check it at the very end (§5) – this is the evidence that “raw CSV files remain immutable”.
* The sentinel is the literal two-character string `\N`. The audit below also found a **second** one – an *empty string* (only in `qualifying.q2/q3`, 2023–24). We convert **only these two tokens** to `NaN` (`na_values=['\\N', ''], keep_default_na=False`). pandas’ default NA list would also silently turn legitimate strings such as `"NA"` or `"None"` into missing values, which we don’t want.
* While reading we also count, per column, how many cells are `\N`, how many are empty strings, and how many are *other* suspicious placeholders (`NA`, `null`, `-` …), so the audit in §2 is based on counts, not on eyeballing.
* We also keep what a **naïve** `pd.read_csv(path)` would infer (dtypes), to measure the impact of semantic typing in §4.

In [ ]:
TABLES = ['circuits','constructor_results','constructor_standings','constructors','driver_standings',
          'drivers','lap_times','pit_stops','qualifying','races','results','seasons','sprint_results','status']
OTHER_TOKENS = [' ', 'NA', 'N/A', 'na', 'null', 'NULL', 'None', 'nan', 'NaN', '-', '--', '?']

def md5(p, chunk=1 << 20):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()

RAW, NAIVE_DTYPES, SENT, EMPTY, OTHER, WS = {}, {}, {}, {}, {}, {}
HASH_BEFORE = {}
t0 = time.time()
for t in TABLES:
    p = RAW_DIR / f'{t}.csv'
    HASH_BEFORE[t] = md5(p)
    s = pd.read_csv(p, dtype=str, keep_default_na=False)                # everything as text → exact token counts
    SENT[t]  = ((s == '\\N') | (s == '')).sum()                          # sentinel = literal \N  OR  empty string
    EMPTY[t] = (s == '').sum()                                            # how many of those are empty strings
    OTHER[t] = s.isin(OTHER_TOKENS).sum()
    WS[t]    = (s != s.apply(lambda c: c.str.strip())).sum()            # leading/trailing whitespace
    del s
    NAIVE_DTYPES[t] = pd.read_csv(p).dtypes                              # what a careless load would give
    RAW[t] = pd.read_csv(p, na_values=['\\N', ''], keep_default_na=False)  # sentinels → NaN, nothing else touched
print(f'loaded {len(TABLES)} tables in {time.time()-t0:.1f}s')
pd.DataFrame({t: {'rows': len(df), 'cols': df.shape[1]} for t, df in RAW.items()}).T

## 2 · Raw audit — EDA **before** cleaning
Goal: find every problem *with code and plots* before touching anything, so each cleaning step in §3 has evidence behind it.
Sub-sections: 2.1 inventory · 2.2 sentinels · 2.3 primary keys · 2.4 foreign keys · 2.5 duplicate grain · 2.6 target & eras · 2.7 grid = 0 · 2.8 qualifying coverage · 2.9 duration formats · 2.10 leakage map.

### 2.1 Inventory — size, columns, memory

In [ ]:
inv = pd.DataFrame({
    t: {'rows': len(df), 'cols': df.shape[1], 'cells': df.size,
        'sentinel cells': int(SENT[t].sum()), 'of which empty-string': int(EMPTY[t].sum()), 'other placeholder cells': int(OTHER[t].sum()),
        'whitespace-padded cells': int(WS[t].sum()),
        'memory MB': df.memory_usage(deep=True).sum() / 1e6}
    for t, df in RAW.items()}).T
inv['% sentinel'] = inv['sentinel cells'] / inv['cells'] * 100
inv

**Reading it:** `lap_times` (~590k rows) and `results` (~27k rows) are the big ones; `results` is the table that becomes our modelling grain. The “of which empty-string”, “other placeholder” and “whitespace” columns tell us whether `\N` is the *only* sentinel to worry about. It is **not**: a few empty-string cells exist (explained in §2.2) – a good example of why we count instead of assuming.

### 2.2 Sentinels — where is `\N` and how much?

In [ ]:
rows = []
for t, df in RAW.items():
    for c in df.columns:
        rows.append((t, c, int(SENT[t][c]), SENT[t][c] / len(df), int(EMPTY[t][c]), int(OTHER[t][c])))
sent = pd.DataFrame(rows, columns=['table', 'column', 'n_sentinel', 'share', 'n_empty_string', 'n_other_placeholders'])
sent_pos = sent[sent.n_sentinel > 0].sort_values('share', ascending=False).reset_index(drop=True)
print(f'{len(sent_pos)} of {len(sent)} columns contain a sentinel (\\N or empty string)')
print('columns with EMPTY-STRING sentinels:', sent[sent.n_empty_string > 0][['table','column','n_empty_string']].values.tolist())
print('columns with OTHER placeholder tokens:', sent[sent.n_other_placeholders > 0][['table','column','n_other_placeholders']].values.tolist())

fig, ax = plt.subplots(figsize=(9, 0.28 * len(sent_pos) + 1.2))
lab = (sent_pos.table + '.' + sent_pos.column)[::-1]
ax.barh(lab, sent_pos.share[::-1], color=BLUE, height=0.7)
ax.xaxis.set_major_formatter(PercentFormatter(1.0)); ax.set_xlim(0, 1.05)
for y, v in enumerate(sent_pos.share[::-1]): ax.text(v + 0.01, y, f'{v:.0%}', va='center', fontsize=8, color=INK2)
ax.set_title('Share of cells that are a sentinel (\\N or empty), by column'); ax.grid(axis='y', visible=False)
save(fig, '01_sentinel_share')

**What the plot says & what we do with it**
* `\N` appears in two *different* ways, and they must be treated differently:
  1. **Structural “not applicable”** – e.g. `qualifying.q2/q3` (driver knocked out earlier), `results.position` (not classified), `results.fastestLap*` (older seasons have no fastest-lap data), `races.fp1…sprint_*` (those sessions did not exist / were not recorded). These are *information*, so we convert to `NaN` but **do not impute** them blindly.
  2. **Completely empty columns** – `constructor_results.status` is `\N` except for 17 rows (see §3.2). It is turned into a boolean flag.
* **A second sentinel hides in the newest data:** `qualifying.q2` / `q3` use an **empty string** (not `\N`) for drivers knocked out earlier in 2023–24 (printed above). A parser that only knew `\N` would have crashed or mis-typed those columns – our duration-parser self-check caught it, and both tokens are now treated as missing.
  No other placeholder token (`NA`, `null`, `-` …) appears (see `n_other_placeholders`).

### 2.3 Primary keys — unique and non-null?

In [ ]:
PKS = {
    'circuits': ['circuitId'], 'constructors': ['constructorId'], 'drivers': ['driverId'], 'races': ['raceId'],
    'seasons': ['year'], 'status': ['statusId'], 'results': ['resultId'], 'qualifying': ['qualifyId'],
    'sprint_results': ['resultId'], 'driver_standings': ['driverStandingsId'],
    'constructor_standings': ['constructorStandingsId'], 'constructor_results': ['constructorResultsId'],
    'pit_stops': ['raceId', 'driverId', 'stop'], 'lap_times': ['raceId', 'driverId', 'lap'],
}
# natural "one row per entity per race" keys we *expect* to be unique too
NATURAL = {
    'results': ['raceId', 'driverId'], 'qualifying': ['raceId', 'driverId'], 'sprint_results': ['raceId', 'driverId'],
    'driver_standings': ['raceId', 'driverId'], 'constructor_standings': ['raceId', 'constructorId'],
    'constructor_results': ['raceId', 'constructorId'],
}
def key_check(df, cols):
    return dict(null_keys=int(df[cols].isna().any(axis=1).sum()), duplicate_keys=int(df.duplicated(cols).sum()))
rows = []
for t, cols in PKS.items():
    rows.append(dict(table=t, key='(' + ', '.join(cols) + ')', kind='primary key', **key_check(RAW[t], cols)))
for t, cols in NATURAL.items():
    rows.append(dict(table=t, key='(' + ', '.join(cols) + ')', kind='natural key (grain)', **key_check(RAW[t], cols)))
pk = pd.DataFrame(rows)
pk['status'] = np.where((pk.null_keys == 0) & (pk.duplicate_keys == 0), 'PASS', 'FAIL')
pk

**Reading it:** every surrogate primary key passes. The *natural* grain key `(raceId, driverId)` is the one to watch: if `results` fails it, then “one row = one driver in one race” is **not** true in the raw data and §2.5/§3.3 must fix it. (All other grain keys should pass.)

### 2.4 Foreign keys — does every reference exist in its parent table?

In [ ]:
FKS = [  # (child, column, parent, parent_column)
    ('results','raceId','races','raceId'), ('results','driverId','drivers','driverId'),
    ('results','constructorId','constructors','constructorId'), ('results','statusId','status','statusId'),
    ('races','circuitId','circuits','circuitId'), ('races','year','seasons','year'),
    ('qualifying','raceId','races','raceId'), ('qualifying','driverId','drivers','driverId'),
    ('qualifying','constructorId','constructors','constructorId'),
    ('sprint_results','raceId','races','raceId'), ('sprint_results','driverId','drivers','driverId'),
    ('sprint_results','constructorId','constructors','constructorId'), ('sprint_results','statusId','status','statusId'),
    ('driver_standings','raceId','races','raceId'), ('driver_standings','driverId','drivers','driverId'),
    ('constructor_standings','raceId','races','raceId'), ('constructor_standings','constructorId','constructors','constructorId'),
    ('constructor_results','raceId','races','raceId'), ('constructor_results','constructorId','constructors','constructorId'),
    ('pit_stops','raceId','races','raceId'), ('pit_stops','driverId','drivers','driverId'),
    ('lap_times','raceId','races','raceId'), ('lap_times','driverId','drivers','driverId'),
]
rows = []
for child, col, parent, pcol in FKS:
    vals = RAW[child][col]
    nn = vals.dropna()
    orphans = int((~nn.isin(RAW[parent][pcol])).sum())
    rows.append(dict(child=child, column=col, parent=f'{parent}.{pcol}', child_rows=len(vals),
                     null_fk=int(vals.isna().sum()), orphans=orphans, status='PASS' if orphans == 0 else 'FAIL'))
fk = pd.DataFrame(rows)
print(f'{(fk.status=="PASS").sum()} / {len(fk)} foreign keys reconcile; total orphans = {fk.orphans.sum()}')
fk

**Reading it:** a `PASS` means every non-null child value exists in the parent’s key – joins will not silently drop rows. Any `FAIL` would need an explicit decision (drop or repair) in §3; there is nothing to repair if all rows are `PASS`.

### 2.5 Grain — who appears more than once in the same race?

In [ ]:
def no_grid_slot(df):
    """Entry that never got a place on the grid: failed to (pre-)qualify / withdrew, or grid=0 with 0 laps.
    (grid=0 with laps>0 is a genuine pit-lane start and is NOT in this group.)"""
    return df.positionText.isin(['F', 'W']) | ((df.grid == 0) & (df.laps == 0))

res = (RAW['results']
       .merge(RAW['races'][['raceId', 'year', 'round', 'circuitId', 'date']], on='raceId', how='left')
       .merge(RAW['status'], on='statusId', how='left'))
res['podium'] = (res.positionOrder <= 3).astype(int)
res['no_grid_slot'] = no_grid_slot(res)

is_dup = res.duplicated(['raceId', 'driverId'], keep=False)
dups = res[is_dup].sort_values(['raceId', 'driverId', 'resultId'])
g = dups.groupby(['raceId', 'driverId'])
pairs = g.agg(n_rows=('resultId', 'size'), n_constructors=('constructorId', 'nunique'), year=('year', 'first'),
              all_never_started=('no_grid_slot', 'all'), any_podium=('podium', 'max'),
              podium_differs=('podium', lambda s: s.nunique() > 1)).reset_index()
print(f'rows involved: {len(dups)}  |  driver-race pairs affected: {len(pairs)}  |  extra rows to remove: {len(dups) - len(pairs)}')
print(f'years affected: {pairs.year.min()}–{pairs.year.max()}  |  rows per pair: {pairs.n_rows.value_counts().to_dict()}')
print(f'pairs where BOTH entries never started (paper duplicates, e.g. DNQ twice): {int(pairs.all_never_started.sum())}')
print(f'pairs with a genuine race entry (shared drive / car swap):               {int((~pairs.all_never_started).sum())}')
print(f'pairs where the two rows disagree on podium (the dangerous ones):         {int(pairs.podium_differs.sum())}')

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
by_year = pairs.groupby('year').size().reindex(range(1950, 1981), fill_value=0)
axes[0].bar(by_year.index, by_year.values, color=BLUE)
axes[0].set_title('Driver-races with >1 result row, by season'); axes[0].set_xlabel('season'); axes[0].set_ylabel('affected driver-races')
kind = pd.Series({'paper duplicate\n(never started)': int(pairs.all_never_started.sum()),
                  'shared drive /\ncar swap': int((~pairs.all_never_started).sum())})
axes[1].bar(kind.index, kind.values, color=[INK2, BLUE], width=0.5)
for x, v in enumerate(kind.values): axes[1].text(x, v + 1, str(v), ha='center', fontsize=9)
axes[1].set_title('What kind of duplicate?'); axes[1].grid(axis='x', visible=False)
save(fig, '02_duplicate_grain')

dups[['raceId','year','driverId','constructorId','grid','positionText','positionOrder','points','laps','status','resultId']].head(14)

**Reading it:** `(raceId, driverId)` is **not** unique in raw `results` – exactly the problem the brief warns about – and *all* duplicates are in the 1950s–1970s (none in the modern era). Two causes: (a) **shared drives / car swaps** (a driver takes over a team-mate’s car, so he gets a second row with a different position), and (b) a few “paper” duplicates (3 pairs) where a driver was entered in two cars and failed to qualify in both. The rule that picks one row is chosen and justified in §3.3 with a comparison of three candidate rules.

### 2.6 The target — podium rate and field size by season/era

In [ ]:
ERA_BINS   = [1949, 1988, 1994, 2008, 2013, 2021, 2024]
ERA_LABELS = ['1950–88', '1989–94', '1995–2008', '2009–13', '2014–21\n(hybrid)', '2022–24']
def era_of(y): return pd.cut(y, ERA_BINS, labels=ERA_LABELS)
res['era'] = era_of(res.year)

season = res.groupby('year').agg(podium_rate=('podium', 'mean'), n_rows=('podium', 'size'), n_races=('raceId', 'nunique'))
season['entrants_per_race'] = season.n_rows / season.n_races
print(f'overall podium rate (raw): {res.podium.mean():.3%}   rows: {len(res):,}')

fig, axes = plt.subplots(2, 1, figsize=(10, 6.2), sharex=True)
axes[0].plot(season.index, season.entrants_per_race, color=BLUE, lw=2)
axes[0].set_title('Entrants per race (raw rows) – the field shrinks and standardises'); axes[0].set_ylabel('entrants / race')
axes[1].plot(season.index, season.podium_rate, color=ORANGE, lw=2)
axes[1].axhline(res.podium.mean(), color=INK2, lw=1, ls='--'); axes[1].text(2024, res.podium.mean() - 0.006, 'overall mean', fontsize=8, color=INK2, ha='right')
axes[1].yaxis.set_major_formatter(PercentFormatter(1.0)); axes[1].set_title('Podium rate (rows with positionOrder ≤ 3) – class imbalance by season'); axes[1].set_xlabel('season')
for a in axes:
    for x in (1989, 1995, 2009, 2014, 2022): a.axvline(x, color=GRID, lw=1, zorder=0)
save(fig, '03_target_and_field_size')
res.groupby('era', observed=True).agg(rows=('podium', 'size'), races=('raceId', 'nunique'), podium_rate=('podium', 'mean'))

**Reading it:** the positive class is rare (≈1 in 8 raw rows) and its rate moves with the era because the *number of entrants changes* (3 podium places over 20 vs 30+ entrants). That is why §3.4 matters: rows for entrants who never started also dilute the rate. It also tells us the **split must be chronological** (Part 2), and that era-sensitive features will be needed.

### 2.7 `grid = 0` — what does it really mean?
The primer says 0 *may* be a pit-lane start (“verify in EDA”). We verify by looking at what those rows did in the race.

In [ ]:
z = res[res.grid == 0].copy()
def kind0(r):
    if 'qualify' in str(r.status).lower(): return 'Did not (pre-)qualify'
    if r.status == 'Withdrew':               return 'Withdrew'
    if r.laps == 0:                          return 'Other, 0 laps (no start)'
    return 'Raced from pit lane / no grid time'
z['kind'] = z.apply(kind0, axis=1)
print(f'grid==0 rows: {len(z):,} ({len(z)/len(res):.1%} of all rows); podium rate among them: {z.podium.mean():.2%}')

dec = res.assign(decade=(res.year // 10) * 10).groupby('decade').apply(lambda d: (d.grid == 0).mean())
comp = z.kind.value_counts()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].bar(dec.index.astype(str) + 's', dec.values, color=BLUE, width=0.7)
axes[0].yaxis.set_major_formatter(PercentFormatter(1.0)); axes[0].set_title('Share of rows with grid = 0, by decade'); axes[0].tick_params(axis='x', rotation=45)
axes[1].barh(comp.index[::-1], comp.values[::-1], color=[ORANGE if 'pit' in k else INK2 for k in comp.index[::-1]], height=0.6)
for y, v in enumerate(comp.values[::-1]): axes[1].text(v + 10, y, f'{v:,}', va='center', fontsize=8)
axes[1].set_title('What the grid = 0 rows actually are'); axes[1].grid(axis='y', visible=False)
save(fig, '04_grid_zero')

**Reading it:** the vast majority of `grid = 0` rows are entrants who **never started** (did not qualify / pre-qualify / withdrew) – they have 0 laps and essentially **never** reach the podium. Only a small minority are genuine pit-lane starts (laps > 0). Therefore `grid = 0` is *not* one thing: treating it as “starting 0th” (i.e. better than pole) would be a serious feature bug, and the never-started rows should not be part of the “will this driver podium?” population (decision in §3.4).

### 2.8 Qualifying coverage per season (the “historical coverage drift”)

In [ ]:
Q = RAW['qualifying']
m = res.merge(Q[['raceId', 'driverId', 'q1', 'q2', 'q3', 'position']].rename(columns={'position': 'q_position'})
              .assign(has_quali=1), on=['raceId', 'driverId'], how='left')
m['has_quali'] = m.has_quali.fillna(0)
cov = m.groupby('year').has_quali.mean()
cov_started = m[~m.no_grid_slot].groupby('year').has_quali.mean()
qy = Q.merge(RAW['races'][['raceId', 'year']], on='raceId')
seg = qy.groupby('year')[['q1', 'q2', 'q3']].agg(lambda s: s.notna().mean())

first_q = int(qy.year.min())
good = cov_started[cov_started >= 0.95]
start_full = next((int(y) for y in good.index if (cov_started.loc[y:] >= 0.95).all()), None)
print(f'qualifying.csv covers seasons {first_q}–{int(qy.year.max())}; first season from which EVERY season has ≥95% of started rows covered: {start_full}')

fig, axes = plt.subplots(1, 2, figsize=(12, 3.9))
axes[0].fill_between(cov.index, cov.values, color=BLUE, alpha=0.18); axes[0].plot(cov.index, cov.values, color=BLUE, lw=2, label='all result rows')
axes[0].plot(cov_started.index, cov_started.values, color=ORANGE, lw=2, label='rows that started the race')
axes[0].yaxis.set_major_formatter(PercentFormatter(1.0)); axes[0].set_ylim(0, 1.05); axes[0].legend(loc='center left')
axes[0].set_title('Share of driver-races that have a qualifying record'); axes[0].set_xlabel('season')
for col, c in zip(['q1', 'q2', 'q3'], [BLUE, ORANGE, AQUA]): axes[1].plot(seg.index, seg[col], color=c, lw=2, label=col)
axes[1].yaxis.set_major_formatter(PercentFormatter(1.0)); axes[1].set_ylim(0, 1.05); axes[1].legend(loc='lower left')
axes[1].set_title('Within qualifying rows: share with q1 / q2 / q3 time'); axes[1].set_xlabel('season')
save(fig, '05_qualifying_coverage')

**Reading it – two different phenomena:**
1. **Coverage drift (left):** `qualifying.csv` starts in 1994, is patchy for ~1996–2002, and is complete (≥95% of started rows) from the season printed above (2003) onward. Before that, `grid` in `results` is the only record of the starting order.
2. **Structural missingness (right):** from 2006 the knockout format means drivers eliminated in Q1 have no `q2/q3` time. This is *real information* (“eliminated in Q1”), not corruption – never fill it with a mean.

**Implication for Part 2:** qualifying-based features are only trustworthy from the season printed above; we will compare (a) restricting the training window, (b) falling back to `grid`, and (c) imputing, with a measured before/after – see §4.4 for the row-count evidence.

### 2.9 Duration / time strings — which formats exist?
Before converting anything to seconds we classify every string, so the parser is written against *observed* formats.

In [ ]:
FORMATS = {'ss.mmm': r'^\d+\.\d+$', 'm:ss.mmm': r'^\d+:\d{2}\.\d+$', 'h:mm:ss.mmm': r'^\d+:\d{2}:\d{2}\.\d+$',
           '+gap (ss.mmm / m:ss / m:ss.mmm)': r'^\+\d+(:\d{2}){0,2}(\.\d+)?$', 'HH:MM:SS clock': r'^\d{2}:\d{2}:\d{2}$'}
DUR_COLS = [('qualifying', 'q1'), ('qualifying', 'q2'), ('qualifying', 'q3'), ('results', 'time'), ('results', 'fastestLapTime'),
            ('sprint_results', 'time'), ('sprint_results', 'fastestLapTime'), ('lap_times', 'time'),
            ('pit_stops', 'duration'), ('pit_stops', 'time'), ('races', 'time')]
rows = []
for t, c in DUR_COLS:
    s = RAW[t][c].dropna().astype(str)
    cnt = {k: int(s.str.match(p).sum()) for k, p in FORMATS.items()}
    cnt['UNRECOGNISED'] = int(len(s) - sum(cnt.values()))
    rows.append(pd.Series(cnt, name=f'{t}.{c}'))
fmt = pd.DataFrame(rows); fmt.insert(0, 'non-null', fmt.sum(axis=1))
fmt

**Reading it:** durations are stored in **several different layouts** (`ss.mmm`, `m:ss.mmm`, `h:mm:ss.mmm`, and `+gap` offsets in `results.time`, one of them even without decimals: `+1:10`), and even *within one column* (`pit_stops.duration` mixes `ss.mmm` with `m:ss.mmm` once a stop lasts a minute or more). A naïve `float()` would crash or – worse – read `1:26.572` wrongly. So §3.2 uses one parser that handles each layout and **counts any value it cannot parse** (should be 0 = `UNRECOGNISED` above).

### 2.10 Leakage map — which columns exist at the prediction cutoff?
The cutoff is **after qualifying, before the race**. This table is our reference for Part 2 (feature engineering) and the report. “R” = the race we predict; “past” = strictly earlier races.

In [ ]:
leak = pd.DataFrame([
    # table, column(s), usable for race R?, note
    ('results', 'grid', 'YES', 'starting slot after penalties; 0 is NOT a position (see §2.7)'),
    ('results', 'driverId, constructorId, number', 'YES', 'identifiers known when the entry list is fixed'),
    ('results', 'position, positionText, positionOrder', 'NO – target / post-race', 'positionOrder<=3 IS the target'),
    ('results', 'points, laps, time, milliseconds', 'NO – post-race', ''),
    ('results', 'fastestLap, rank, fastestLapTime, fastestLapSpeed', 'NO – post-race', ''),
    ('results', 'statusId', 'NO – post-race', 'why the race ended; only used for Q3 (mechanical retirements)'),
    ('results (earlier races)', 'any of the above, aggregated', 'YES – only strictly earlier races', 'basis for leakage-safe form features'),
    ('qualifying', 'position, q1, q2, q3', 'YES', 'recorded before the race; coverage drifts (§2.8)'),
    ('races', 'year, round, circuitId, date, time, name', 'YES', 'schedule is public in advance'),
    ('races', 'fp*_date/time, quali_date/time, sprint_date/time', 'YES (low value)', 'schedule only'),
    ('driver_standings / constructor_standings', 'row for race R', 'NO – post-race', 'cumulative INCLUDING race R; must be shifted to “before R”'),
    ('driver_standings / constructor_standings', 'row for the previous race', 'YES', 'standings entering race R'),
    ('constructor_results', 'any for race R', 'NO – post-race', ''),
    ('pit_stops, lap_times', 'race R', 'NO – post-race', 'only earlier races could feed history features'),
    ('sprint_results', 'same weekend', 'JUDGEMENT CALL', 'technically pre-race, but only ~360 rows (2021+) → sparse'),
    ('drivers', 'dob, nationality', 'YES', 'age at race date; nationality ↔ circuit country for home race (Q2)'),
    ('circuits', 'country, lat, lng, alt', 'YES', 'static'),
], columns=['table', 'columns', 'usable_for_race_R', 'note'])
leak

## 3 · Cleaning
**Principles** – (1) raw tables are never modified; every step works on `CLEAN = {t: df.copy()}`; (2) every step prints what it changed; (3) nothing is *dropped* without a rule that is justified by the audit above and logged in §5.

In [ ]:
CLEAN = {t: df.copy() for t, df in RAW.items()}
LOG = []          # (step, rows_before, rows_after, note)
def log(step, before, after, note=''): LOG.append(dict(step=step, rows_before=before, rows_after=after, note=note))

### 3.1 Sentinels → `NaN` (already done at load, now **verified**)
At load we converted only the two sentinel tokens (`\N` and the empty string). Here we prove none survives in any cleaned table.

In [ ]:
left = {t: int(df.select_dtypes(include=['object', 'string']).isin(['\\N', '']).sum().sum()) for t, df in CLEAN.items()}
print('remaining sentinel strings (\\N or empty) per table:', {k: v for k, v in left.items() if v})
assert sum(left.values()) == 0, 'sentinel still present'
print('OK – 0 sentinel strings remain in', len(CLEAN), 'tables. Total NaN cells:', int(sum(df.isna().sum().sum() for df in CLEAN.values())))

### 3.2 Semantic typing — dates, times, durations → real types
**Why:** strings cannot be compared, sorted chronologically, subtracted or fed to a model. We need real `datetime`, `timedelta` and numeric-seconds values.
* **One duration parser** for all observed layouts (`ss.mmm`, `m:ss.mmm`, `h:mm:ss.mmm`, `+gap`). It returns `NaN` (and counts a failure) for anything else instead of guessing.
* **Self-check:** `lap_times.time` and `pit_stops.duration` also exist as integer milliseconds in the same rows, so we can verify our parser against an independent source.
* `constructor_results.status` (17 non-null rows, all `D`) becomes a boolean flag (see below).

In [ ]:
_DUR = re.compile(r'^(?:(\d+):)?(?:(\d+):)?(\d+(?:\.\d+)?)$')
def _one(x):
    x = x.strip().lstrip('+')
    m = _DUR.match(x)
    if not m: return np.nan
    a, b, c = m.groups()
    sec = float(c)
    if a is not None and b is None:  sec += 60 * int(a)                       # m:ss.mmm
    elif a is not None and b is not None: sec += 3600 * int(a) + 60 * int(b)  # h:mm:ss.mmm
    return sec

PARSE_FAILURES = {}
def to_seconds(s, name):
    s = s.astype('string')
    uniq = s.dropna().unique()
    mp = {u: _one(u) for u in uniq}                       # parse each distinct string once (fast on 590k laps)
    out = pd.to_numeric(s.map(mp), errors='coerce').astype('float64')
    PARSE_FAILURES[name] = int((s.notna() & out.isna()).sum())
    return out

# ---- races: dates → datetime64, clock times → timedelta, plus one combined start timestamp -----------------
r = CLEAN['races']
for p in ['', 'fp1_', 'fp2_', 'fp3_', 'quali_', 'sprint_']:
    r[f'{p}date'] = pd.to_datetime(r[f'{p}date'], errors='coerce')
    r[f'{p}time'] = pd.to_timedelta(r[f'{p}time'], errors='coerce')
r['race_start_datetime'] = r['date'] + r['time']        # NaT when the clock time is unknown. Time zone: see note below (not asserted as UTC)
# ---- drivers ----------------------------------------------------------------------------------------------
d = CLEAN['drivers']; d['dob'] = pd.to_datetime(d['dob'], errors='coerce'); d['number'] = d['number'].astype('Int64')
# ---- qualifying: q1/q2/q3 → seconds ------------------------------------------------------------------------
q = CLEAN['qualifying']
for c in ['q1', 'q2', 'q3']: q[f'{c}_s'] = to_seconds(q[c], f'qualifying.{c}')
q['number'] = q['number'].astype('Int64')
# ---- results / sprint_results -----------------------------------------------------------------------------
for t in ['results', 'sprint_results']:
    x = CLEAN[t]
    for c in ['position', 'number', 'fastestLap'] + (['rank'] if t == 'results' else []): x[c] = x[c].astype('Int64')
    x['fastestLapTime_s'] = to_seconds(x['fastestLapTime'], f'{t}.fastestLapTime')
    x['time_s'] = to_seconds(x['time'], f'{t}.time')                       # winner = total time, others = gap to winner
    x['time_is_gap'] = x['time'].astype('string').str.startswith('+').fillna(False).astype(bool)
    if 'fastestLapSpeed' in x: x['fastestLapSpeed'] = pd.to_numeric(x['fastestLapSpeed'], errors='coerce')
# ---- lap_times & pit_stops ---------------------------------------------------------------------------------
lt = CLEAN['lap_times']; lt['time_s'] = to_seconds(lt['time'], 'lap_times.time')
ps = CLEAN['pit_stops']; ps['duration_s'] = to_seconds(ps['duration'], 'pit_stops.duration'); ps['clock'] = pd.to_timedelta(ps['time'], errors='coerce')
# ---- constructor_results.status: sparse code → explicit boolean --------------------------------------------
cr = CLEAN['constructor_results']
print('constructor_results.status values:', cr.status.value_counts(dropna=False).to_dict())
cr['constructor_disqualified'] = cr['status'].eq('D'); cr.drop(columns='status', inplace=True)

print('parse failures (value present but not parsed):', {k: v for k, v in PARSE_FAILURES.items()} )
assert sum(PARSE_FAILURES.values()) == 0, 'unparsed duration strings – extend the parser'

# ---- independent checks of the parser ----------------------------------------------------------------------
chk_lap = (lt.time_s * 1000 - lt.milliseconds).abs().max()
chk_pit = (ps.duration_s * 1000 - ps.milliseconds).abs().max()
w = CLEAN['results']; w = w[~w.time_is_gap & w.time_s.notna() & w.milliseconds.notna()]
chk_res = (w.time_s * 1000 - w.milliseconds).abs()
print(f'max |parsed - milliseconds|  lap_times: {chk_lap:.3f} ms | pit_stops: {chk_pit:.3f} ms | results winner times: {chk_res.max():.3f} ms (n={len(w):,}, >1ms off: {(chk_res>1).sum()})')

**A note on the combined start time:** `race_start_datetime` is simply the race `date` plus its `time`; the files do not state a time zone, so we do **not** call it UTC. The values are consistent with UTC (e.g. 2009: Melbourne 06:00 ≙ 17:00 local UTC+11, Silverstone 12:00 ≙ 13:00 local UTC+1) – but that comes from known local start times, not from documentation, so the column name stays neutral. It is only used for ordering/spacing events, never as a feature, and only ~51% of 2000s races and none before 2000 have a time at all (we use `date` for chronology).

**Result:** all duration strings parse (0 failures). The parsed values match the independent millisecond columns **exactly** for all 589,081 laps and all 11,371 pit stops, and for all but **1 of 1,128** race-winner times (a ~0.5 s inconsistency *inside the source file* – we keep the data as given). So the parser is trustworthy. Dates are real `datetime64`, clock times are `timedelta`, and `constructor_results.status` is now a clear boolean (`constructor_disqualified`; the few flagged rows are listed in §3.5 – irrelevant for our target, and the column must not be used as a feature because it is post-race).

### 3.3 Grain — exactly one row per `(raceId, driverId)`
**Candidate rules** for choosing which row to keep when a driver has several rows in one race. We compute what each rule would do *before* choosing:
| rule | idea |
|---|---|
| `best_position` | keep the row with the best (lowest) `positionOrder`; ties → most laps → lowest `resultId` |
| `most_laps` | keep the row where the driver covered the most laps |
| `first_result` | keep the lowest `resultId` (the original entry) |

In [ ]:
R = (CLEAN['results']
     .merge(CLEAN['races'][['raceId', 'year', 'round', 'circuitId', 'date']], on='raceId', how='left')
     .merge(CLEAN['status'], on='statusId', how='left'))
R['no_grid_slot'] = no_grid_slot(R)
KEY = ['raceId', 'driverId']
RULES = {   # sort spec → first row per pair is the one kept
    'best_position': (['positionOrder', 'laps', 'resultId'], [True, False, True]),
    'most_laps':     (['laps', 'positionOrder', 'resultId'], [False, True, True]),
    'first_result':  (['resultId'], [True]),
}
def pick(df, rule):
    cols, asc = RULES[rule]
    return df.sort_values(KEY + cols, ascending=[True, True] + asc).groupby(KEY).head(1)

D = R[R.duplicated(KEY, keep=False)]
kept = {rule: pick(D, rule) for rule in RULES}
base = set(kept['best_position'].resultId)
cmp_rules = pd.DataFrame({
    rule: {'pairs resolved': len(k), 'rows kept that are podium': int((k.positionOrder <= 3).sum()),
           'kept rows with grid = 0': int((k.grid == 0).sum()), 'kept rows that finished (laps>0)': int((k.laps > 0).sum()),
           'pairs where choice ≠ best_position': int((~k.resultId.isin(base)).sum())}
    for rule, k in kept.items()}).T
print(cmp_rules.to_string())

**Why `best_position` is the default (and what it costs):**
* In real F1 a **shared car is credited to both drivers** – if the shared car finished 3rd, both drivers are on the podium. Our label “does this driver finish on the podium?” should follow the official credit, which is the driver’s **best recorded result** in that race.
* `first_result` can *lose* a podium the driver was officially credited with: it keeps only 6 podium rows vs 22 for `best_position` (table above). `most_laps` happens to pick exactly the same rows as `best_position` here (0 pairs differ), so the choice between those two is cosmetic; we keep `best_position` because its justification (official credit) is the more defensible one.
* **Cost / honesty:** after a car swap the kept row’s `grid` may be the *car’s* starting slot rather than the driver’s own. This only affects the handful of pairs below, **all in 1950–1978**. If Part 2 restricts training to the modern era (as the qualifying-coverage evidence in §2.8 suggests), these rows never reach the model – so the choice is low-risk either way. Flip `CFG['dup_rule']` to measure it.

In [ ]:
n_before = len(R)
R_sorted = R.sort_values(KEY + RULES[CFG['dup_rule']][0], ascending=[True, True] + RULES[CFG['dup_rule']][1])
R = R_sorted.drop_duplicates(KEY, keep='first').sort_values('resultId').reset_index(drop=True)
DROPPED = R_sorted[R_sorted.duplicated(KEY, keep='first')].sort_values('resultId')
assert not R.duplicated(KEY).any(), 'grain still violated'
log(f"dedupe (raceId, driverId) – rule '{CFG['dup_rule']}'", n_before, len(R), f'{len(DROPPED)} duplicate rows removed')
print(f"rule = {CFG['dup_rule']}: {n_before:,} → {len(R):,} rows  ({len(DROPPED)} removed)  |  (raceId, driverId) unique: {not R.duplicated(KEY).any()}")
DROPPED[['raceId','year','driverId','constructorId','grid','positionText','positionOrder','laps','status','resultId']].head(8)

### 3.4 Population — who is a legitimate “driver entering a race”?
`results` contains entries that **never got a place on the grid** (did not qualify / pre-qualify, withdrew). The question we predict is *“will this driver, who is lining up on the grid, finish on the podium?”* – the cutoff is “after qualifying”, and by then it is already known that these entries are not racing.
Rule (`no_grid_slot`): `positionText ∈ {F, W}` **or** (`grid = 0` **and** `laps = 0`). Genuine **pit-lane starts** (`grid = 0` but `laps > 0`) are *kept*.

In [ ]:
chk = pd.crosstab(R.positionText.where(R.positionText.isin(['F', 'W', 'N', 'D', 'E', 'R']), 'classified (digit)'),
                  np.where(R.grid == 0, 'grid = 0', 'grid > 0'))
chk['rows'] = chk.sum(axis=1)
print(chk.to_string())
n0 = len(R)
R['podium'] = (R.positionOrder <= 3).astype(int)
R['started'] = ~R.no_grid_slot
print(f"\nno grid slot: {int(R.no_grid_slot.sum()):,} rows ({R.no_grid_slot.mean():.1%}); of these podium: {int(R[R.no_grid_slot].podium.sum())}")
print(f"pit-lane starts kept (grid=0, laps>0): {int(((R.grid == 0) & (R.laps > 0)).sum()):,}")
ex = R[R.no_grid_slot]
print(f"of the excluded rows: {int((ex.grid == 0).sum()):,} have grid = 0 (never on the grid), but {int((ex.grid > 0).sum()):,} DID have a grid slot (qualified, then withdrew / failed to start)")
MODEL_BASE = R[R.started].copy() if CFG['exclude_non_starters'] else R.copy()
log('exclude entries with no grid slot' if CFG['exclude_non_starters'] else 'keep non-starters (CFG)', n0, len(MODEL_BASE),
    f"{n0 - len(MODEL_BASE):,} rows removed" if CFG['exclude_non_starters'] else 'no rows removed')
print(f'MODEL_BASE rows: {len(MODEL_BASE):,}   podium rate: {MODEL_BASE.podium.mean():.3%}   (before: {R.podium.mean():.3%})')

**Why exclude them:** (1) they are not part of the real-world question; (2) they carry **no signal for the model** – their podium rate is 0 by construction, so they would only inflate the majority class and make every metric look better than it is; (3) they distort the era comparison (1950s–80s had many never-started entries). **Honest caveat:** the filter uses `laps`/`positionText`, which are post-race columns. For most excluded rows it merely reconstructs a fact known at the cutoff (“no grid slot”). But a minority (printed above, `grid > 0`) qualified and *then* withdrew – for them the exclusion uses after-the-fact knowledge. They can never podium, so including them would only add unpredictable zeros; the effect is small and testable via `CFG['exclude_non_starters']`. These columns never enter the feature set.

### 3.5 Sanity / range checks on the cleaned tables
We **report** out-of-range values rather than silently dropping them – only things that feed the target or features need action.

In [ ]:
Rb = MODEL_BASE
checks = [
    ('results: positionOrder < 1', int((R.positionOrder < 1).sum())),
    ('results: grid < 0', int((R.grid < 0).sum())),
    ('results: laps < 0', int((R.laps < 0).sum())),
    ('results: points < 0', int((R.points < 0).sum())),
    ('results: date missing after join', int(R.date.isna().sum())),
    ('drivers: dob after 2010 / before 1850', int(((CLEAN['drivers'].dob > '2010-01-01') | (CLEAN['drivers'].dob < '1850-01-01')).sum())),
    ('qualifying: q time outside 30–200 s', int(((CLEAN['qualifying'][['q1_s','q2_s','q3_s']].stack() < 30) | (CLEAN['qualifying'][['q1_s','q2_s','q3_s']].stack() > 200)).sum())),
    ('lap_times: lap < 40 s or > 300 s (red-flag / SC laps possible)', int(((lt.time_s < 40) | (lt.time_s > 300)).sum())),
    ('pit_stops: duration > 120 s (red-flag stops)', int((ps.duration_s > 120).sum())),
]
display_chk = pd.DataFrame(checks, columns=['check', 'rows flagged'])
print(display_chk.to_string(index=False))

pod_per_race = Rb.groupby('raceId').podium.sum()
vc = pod_per_race.value_counts().sort_index()
yr_of = Rb.groupby('raceId').year.first()
odd = pod_per_race[pod_per_race != 3]
print(f"\npodium rows per race (modelling base): {vc.to_dict()}")
print(f"races ≠ 3 podium rows: {len(odd)} → seasons {int(yr_of[odd.index].min())}–{int(yr_of[odd.index].max())}" if len(odd) else 'all races have exactly 3 podium rows')
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.bar(vc.index.astype(str), vc.values, color=[ORANGE if i == 3 else INK2 for i in vc.index], width=0.6)
for x, v in enumerate(vc.values): ax.text(x, v + vc.max() * 0.03, f'{v:,}', ha='center', fontsize=8)
ax.set_title('Podium rows per race after cleaning (3 expected)'); ax.set_xlabel('podium rows in the race'); ax.set_ylabel('races'); ax.grid(axis='x', visible=False)
save(fig, '06_podium_rows_per_race')

**Reading it:** the range checks flag no impossible values in the columns that matter (`positionOrder`, `grid`, `laps`, `points`). Unusually long laps/pit stops are real (safety cars, red flags) and those tables are post-race anyway. Almost every race has exactly **3 podium rows** – the few exceptions are old shared-car results where two drivers are credited with the same finishing place; they are all in 1950–1960 and are documented here, not “fixed”.

### 3.5b Text labels → inspect, fix, then verify
Types and keys can be perfect while the *text* is still inconsistent (a trailing space, two spellings of one country). That silently breaks joins and groupings – and Question 2 (home race = driver nationality ↔ circuit country) depends on exactly these labels. So: first inspect, then fix, then run the full check-suite in §3.5c.

In [ ]:
# ---- inspect: padded cells and near-duplicate labels ------------------------------------------------------
import difflib
print('whitespace-padded cells per table (from the load step):', {t: int(WS[t].sum()) for t in TABLES if WS[t].sum()})
for t, c in [('drivers', 'nationality'), ('constructors', 'nationality'), ('circuits', 'country')]:
    vals = sorted(CLEAN[t][c].dropna().unique())
    near = {v: difflib.get_close_matches(v.strip(), [w for w in vals if w != v], n=2, cutoff=0.8) for v in vals}
    near = {k: v for k, v in near.items() if v}
    print(f'\n{t}.{c}: {len(vals)} distinct values. Near-duplicates: {near}')
print('\nraw repr of padded driver nationalities:', [repr(v) for v in CLEAN['drivers'].nationality.unique() if v != v.strip()])
print('circuits whose country is spelled "United States":', CLEAN['circuits'][CLEAN['circuits'].country == 'United States'].name.tolist())
print('circuits whose country is spelled "USA"          :', CLEAN['circuits'][CLEAN['circuits'].country == 'USA'].name.tolist())

# ---- systematic scan of EVERY short text column (not just the ones we thought of) --------------------------------------
rows = []
for t, df in CLEAN.items():
    for c in df.select_dtypes(include=['object', 'string']).columns:
        v = df[c].dropna()
        if v.nunique() > 300 or c == 'url': continue                      # skip free-text / link / high-cardinality columns
        norm = v.str.lower().str.replace(r'[^a-z0-9]', '', regex=True)
        rows.append((t, c, v.nunique(), v.nunique() - norm.nunique(), int((v != v.str.strip()).sum())))
scan = pd.DataFrame(rows, columns=['table', 'column', 'distinct values', 'case/punctuation collisions', 'padded cells'])
print(f'\nscanned {len(scan)} short text columns; columns with a collision or padded cell:')
print(scan[(scan['case/punctuation collisions'] > 0) | (scan['padded cells'] > 0)].to_string(index=False) or 'none')

**What the inspection shows:** (1) one driver nationality has a **trailing space and a different spelling** – `'Argentinian '` (Franco Colapinto, 2024) vs `'Argentine'` for all other Argentine drivers; (2) the United States appears as **both `USA` and `United States`** in `circuits.country`. Left alone, an Argentine-driver-at-Argentine-circuit lookup and an American-driver-at-US-circuit lookup would each silently miss rows.

**Fix (kept minimal and logged):** strip leading/trailing spaces in every text column, then map the two variant spellings to the canonical one used by the rest of the table. We do **not** rewrite anything else.

In [ ]:
# ---- fix -----------------------------------------------------------------------------------------------------------------
n_stripped = 0
for t, df in CLEAN.items():
    for c in df.select_dtypes(include=['object', 'string']).columns:
        s_ = df[c]; m_ = s_.notna() & (s_ != s_.str.strip())
        n_stripped += int(m_.sum()); df[c] = s_.str.strip()
SPELLING_FIX = {('drivers', 'nationality'): {'Argentinian': 'Argentine'}, ('circuits', 'country'): {'United States': 'USA'}}
n_mapped = 0
for (t, c), mp in SPELLING_FIX.items():
    n_mapped += int(CLEAN[t][c].isin(mp).sum()); CLEAN[t][c] = CLEAN[t][c].replace(mp)
print(f'stripped {n_stripped} padded cells; remapped {n_mapped} variant spellings → {SPELLING_FIX}')
log('text labels: strip padding + 2 spelling variants', len(CLEAN['drivers']), len(CLEAN['drivers']), f'row counts unchanged; {n_stripped} padded cell(s) stripped, {n_mapped} cells remapped: drivers.nationality Argentinian→Argentine, circuits.country United States→USA')

### 3.5c Table-by-table integrity checks (types · ranges · labels · cross-table consistency)
Section 2 audited keys, foreign keys and placeholders. Here we go **table by table** and check what is still unverified: are the ID/number columns really numeric, are values in a valid range, are text labels consistent, and do tables agree with each other? Every check writes one row to a findings table – `PASS` (nothing flagged), `INFO` (flagged but harmless/expected), or `WARN` (needs a decision).

In [ ]:
CHK = []
def chk(table, check, n_bad, note='', level='WARN'):
    CHK.append(dict(table=table, check=check, flagged=int(n_bad), status='PASS' if int(n_bad) == 0 else level, note=note))

# ---- 1. numeric typing: ID/count columns must be whole numbers with no nulls; measures must be numeric ----------
INT_COLS = {
    'circuits': ['circuitId'], 'constructors': ['constructorId'], 'seasons': ['year'], 'status': ['statusId'],
    'driver_standings': ['driverStandingsId', 'raceId', 'driverId', 'position', 'wins'],
    'constructor_standings': ['constructorStandingsId', 'raceId', 'constructorId', 'position', 'wins'],
    'constructor_results': ['constructorResultsId', 'raceId', 'constructorId'],
    'races': ['raceId', 'year', 'round', 'circuitId'], 'drivers': ['driverId'],
    'qualifying': ['qualifyId', 'raceId', 'driverId', 'constructorId', 'position'],
    'results': ['resultId', 'raceId', 'driverId', 'constructorId', 'grid', 'positionOrder', 'laps', 'statusId'],
    'sprint_results': ['resultId', 'raceId', 'driverId', 'constructorId', 'grid', 'positionOrder', 'laps', 'statusId'],
    'lap_times': ['raceId', 'driverId', 'lap', 'position', 'milliseconds'],
    'pit_stops': ['raceId', 'driverId', 'stop', 'lap', 'milliseconds'],
}
FLOAT_COLS = {'circuits': ['lat', 'lng', 'alt'], 'driver_standings': ['points'], 'constructor_standings': ['points'],
              'constructor_results': ['points'], 'results': ['points'], 'sprint_results': ['points'],
              'pit_stops': ['duration_s'], 'lap_times': ['time_s']}
for t, cols in INT_COLS.items():
    bad = 0
    for c in cols:
        v = pd.to_numeric(CLEAN[t][c], errors='coerce')
        bad += int(v.isna().sum() + ((v.dropna() % 1) != 0).sum())
    chk(t, f'integer columns are whole numbers, no nulls ({len(cols)} cols)', bad)
for t, cols in FLOAT_COLS.items():
    bad = sum(int(pd.to_numeric(CLEAN[t][c], errors='coerce').isna().sum()) for c in cols)
    chk(t, f'numeric columns parse ({", ".join(cols)})', bad)

# ---- 2. circuits ------------------------------------------------------------------------------------------------
ci = CLEAN['circuits']
chk('circuits', 'lat outside [-90, 90] / lng outside [-180, 180]', int(((ci.lat.abs() > 90) | (ci.lng.abs() > 180)).sum()))
chk('circuits', 'alt outside [-500, 4000] m', int(((ci.alt < -500) | (ci.alt > 4000)).sum()))
chk('circuits', 'duplicate circuitRef / duplicate name', int(ci.circuitRef.duplicated().sum() + ci.name.duplicated().sum()))
chk('circuits', 'text padded with spaces (name, location, country)', int(sum((ci[c] != ci[c].str.strip()).sum() for c in ['name', 'location', 'country'])))
chk('circuits', 'country spelled in several case variants', int(ci.country.nunique() - ci.country.str.lower().nunique()))
SYN_COUNTRY = [{'USA', 'United States'}, {'UK', 'United Kingdom'}, {'UAE', 'United Arab Emirates'}, {'Korea', 'South Korea'}, {'Netherlands', 'Holland'}]
chk('circuits', 'same country under two names (USA/United States, UK/United Kingdom, …)', sum(s_ <= set(ci.country) for s_ in SYN_COUNTRY))

# ---- 3. constructors / status / seasons -------------------------------------------------------------------------
co = CLEAN['constructors']
chk('constructors', 'duplicate constructorRef', int(co.constructorRef.duplicated().sum()))
chk('constructors', 'duplicate constructor name (different ids, same name)', int(co.name.duplicated().sum()), 'a team that renamed keeps separate ids; same *name* twice would be suspicious', 'INFO')
chk('constructors', 'text padded with spaces / nationality case variants', int(sum((co[c] != co[c].str.strip()).sum() for c in ['name', 'nationality']) + co.nationality.nunique() - co.nationality.str.lower().nunique()))
st = CLEAN['status']
chk('status', 'duplicate status labels (exact or case-insensitive)', int(st.status.str.lower().str.strip().duplicated().sum()))
se = CLEAN['seasons']
chk('seasons', 'missing years between first and last season', int(len(set(range(se.year.min(), se.year.max() + 1)) - set(se.year))))
chk('seasons', 'url missing, not http(s), or not containing the year', int((~se.apply(lambda r: str(r.url).startswith('http') and str(r.year) in str(r.url), axis=1)).sum()))

# ---- 4. races -----------------------------------------------------------------------------------------------------
ra = CLEAN['races']
chk('races', 'missing / invalid race date', int(ra.date.isna().sum()))
chk('races', 'date year ≠ year column', int((ra.date.dt.year != ra.year).sum()))
chk('races', 'duplicate (year, round)', int(ra.duplicated(['year', 'round']).sum()))
g = ra.groupby('year')
chk('races', 'seasons whose rounds are not 1..N without gaps', int((g['round'].max() != g.size()).sum()))
ordered = ra.sort_values(['year', 'round']).groupby('year').date.apply(lambda s: s.is_monotonic_increasing)
chk('races', 'seasons where round order ≠ date order', int((~ordered).sum()), 'would break "previous race" logic for form features')
chk('races', 'races with no result rows', int((~ra.raceId.isin(CLEAN['results'].raceId)).sum()))

# ---- 5. drivers ----------------------------------------------------------------------------------------------------
dv = CLEAN['drivers']
chk('drivers', 'duplicate driverRef', int(dv.driverRef.duplicated().sum()))
chk('drivers', 'code present but not 3 capital letters', int((dv.code.dropna().str.fullmatch(r'[A-Z]{3}') == False).sum()))
chk('drivers', 'empty forename / surname', int((dv.forename.str.strip() == '').sum() + (dv.surname.str.strip() == '').sum()))
chk('drivers', 'text padded with spaces (names, nationality)', int(sum((dv[c] != dv[c].str.strip()).sum() for c in ['forename', 'surname', 'nationality'])))
chk('drivers', 'nationality spelled in several case variants', int(dv.nationality.nunique() - dv.nationality.str.lower().nunique()))
SYN_NAT = [{'Argentine', 'Argentinian'}, {'Dutch', 'Netherlander'}, {'Monegasque', 'Monacan'}, {'Czech', 'Czechoslovakian'}]
chk('drivers', 'same nationality under two spellings (Argentine/Argentinian, …)', sum(s_ <= set(dv.nationality) for s_ in SYN_NAT))
chk('drivers', 'same forename + surname + dob twice', int(dv.duplicated(['forename', 'surname', 'dob']).sum()))
age = (R.date - R.merge(dv[['driverId', 'dob']], on='driverId', how='left').dob).dt.days / 365.25
chk('drivers', 'age at race outside 16–65 years', int(((age < 16) | (age > 65)).sum()), 'age will be a feature – implausible values matter', 'WARN')
chk('drivers', 'permanent number missing (share of drivers)', int(dv.number.isna().sum()), 'numbers only exist for recent drivers – not used as a feature', 'INFO')

# ---- 6. standings & constructor_results --------------------------------------------------------------------
for t in ['driver_standings', 'constructor_standings']:
    x = CLEAN[t]
    chk(t, 'points < 0, position < 1 or wins < 0', int(((x.points < 0) | (x.position < 1) | (x.wins < 0)).sum()))
    chk(t, 'positionText disagrees with position', int((x.positionText.astype(str) != x.position.astype(str)).sum()), 'letters (e.g. "-" / "D") are legitimate for non-numeric standings', 'INFO')
    chk(t, 'tied positions inside one race', int(x.duplicated(['raceId', 'position']).sum()), 'ties on points are broken by wins, so duplicates are unexpected')
x = CLEAN['constructor_results']
chk('constructor_results', 'points < 0', int((x.points < 0).sum()))
_ra = CLEAN['races'].set_index('raceId'); _lab = lambda ids: ', '.join(f'{k} ({c})' for k, c in ids.map(lambda r: f"{int(_ra.year[r])} {_ra['name'][r]}").value_counts().head(3).items())
chk('constructor_results', 'rows flagged constructor_disqualified', int(x.constructor_disqualified.sum()), 'races: ' + _lab(x[x.constructor_disqualified].raceId) + ' — a post-race flag, never a feature', 'INFO')

# ---- 7. qualifying ---------------------------------------------------------------------------------------------
q = CLEAN['qualifying']
chk('qualifying', 'position < 1', int((q.position < 1).sum()))
chk('qualifying', 'two drivers with the same qualifying position in one race', int(q.duplicated(['raceId', 'position']).sum()))
chk('qualifying', 'q2 present but q1 missing, or q3 present but q2 missing', int(((q.q2_s.notna() & q.q1_s.isna()) | (q.q3_s.notna() & q.q2_s.isna())).sum()), 'q2/q3 gaps are legitimate eliminations – only the reverse order would be an error')
key_res = set(zip(R.raceId, R.driverId))
chk('qualifying', '(raceId, driverId) not found in results', int((~pd.Series(list(zip(q.raceId, q.driverId))).isin(key_res)).sum()))
m_q = q.merge(R[['raceId', 'driverId', 'constructorId']], on=['raceId', 'driverId'], how='inner', suffixes=('', '_res'))
chk('qualifying', 'constructor differs from the one in results', int((m_q.constructorId != m_q.constructorId_res).sum()), 'driver changed team between qualifying and race', 'INFO')

# ---- 8. results / sprint_results ---------------------------------------------------------------------------
for t, base in [('results', R), ('sprint_results', CLEAN['sprint_results'])]:
    x = base
    is_num = x.positionText.astype(str).str.fullmatch(r'\d+')
    chk(t, 'position missing although positionText is a number (or the reverse)', int((x.position.isna() == is_num).sum()), 'source quirk in: ' + _lab(x[x.position.isna() == is_num].raceId) + '; the target uses positionOrder, which is always filled', 'INFO')
    chk(t, 'positionOrder < 1, grid < 0, laps < 0 or points < 0', int(((x.positionOrder < 1) | (x.grid < 0) | (x.laps < 0) | (x.points < 0)).sum()))
n_ent = R.groupby('raceId').raceId.transform('size')
chk('results', 'grid larger than the number of entrants in that race', int((R.grid > n_ent).sum()), f"seasons {int(R[R.grid > n_ent].raceId.map(_ra.year).min())}–{int(R[R.grid > n_ent].raceId.map(_ra.year).max())} (incomplete old entry lists); outside the modelling window", 'INFO')
chk('results', '(raceId, driverId) duplicates AFTER our de-duplication', int(R.duplicated(KEY).sum()), 'validates the §3.3 decision')
sp = CLEAN['sprint_results']
chk('sprint_results', 'duplicate (raceId, driverId)', int(sp.duplicated(KEY).sum()))
chk('sprint_results', 'sprint driver has no Sunday race row', int((~pd.Series(list(zip(sp.raceId, sp.driverId))).isin(key_res)).sum()))
sprint_race_ids = set(sp.raceId)
chk('sprint_results', 'sprint race whose races.sprint_date is empty', int(ra[ra.raceId.isin(sprint_race_ids)].sprint_date.isna().sum()), 'schedule columns are patchy even for recent seasons', 'INFO')

# ---- 9. lap_times & pit_stops ----------------------------------------------------------------------------------
lt = CLEAN['lap_times']
chk('lap_times', 'lap < 1, position < 1 or milliseconds ≤ 0', int(((lt.lap < 1) | (lt.position < 1) | (lt.milliseconds <= 0)).sum()))
gl = lt.groupby(['raceId', 'driverId']).lap.agg(['max', 'size'])
chk('lap_times', 'driver-races whose lap numbers have gaps', int((gl['max'] != gl['size']).sum()))
chk('lap_times', '(raceId, driverId) not found in results', int((~pd.Series(list(gl.index)).isin(key_res)).sum()))
ps = CLEAN['pit_stops']
chk('pit_stops', 'stop < 1, lap < 1 or milliseconds ≤ 0', int(((ps.stop < 1) | (ps.lap < 1) | (ps.milliseconds <= 0)).sum()))
gp = ps.groupby(['raceId', 'driverId']).stop.agg(['max', 'size'])
chk('pit_stops', 'driver-races whose stop numbers have gaps', int((gp['max'] != gp['size']).sum()), 'races: ' + _lab(gp[gp['max'] != gp['size']].reset_index().raceId) + '; post-race table, never a feature', 'INFO')
chk('pit_stops', 'duration_s disagrees with milliseconds by > 1 ms', int(((ps.duration_s * 1000 - ps.milliseconds).abs() > 1).sum()))
pl = ps.merge(R[['raceId', 'driverId', 'laps']], on=['raceId', 'driverId'], how='left')
chk('pit_stops', 'pit-stop lap beyond the laps the driver completed', int((pl.lap > pl.laps).sum()), 'pit stop on the lap of retirement can be recorded one lap late', 'INFO')

findings = pd.DataFrame(CHK)
print(findings.status.value_counts().to_dict())
findings[findings.status != 'PASS'][['table', 'check', 'flagged', 'status', 'note']].reset_index(drop=True)

In [ ]:
summ = findings.pivot_table(index='table', columns='status', values='check', aggfunc='count', fill_value=0)
for c in ['PASS', 'INFO', 'WARN']:
    if c not in summ: summ[c] = 0
summ = summ[['PASS', 'INFO', 'WARN']].sort_values(['WARN', 'INFO'])
fig, ax = plt.subplots(figsize=(8, 4.8)); acc = np.zeros(len(summ))
for col, colr in [('PASS', BLUE), ('INFO', INK2), ('WARN', ORANGE)]:
    ax.barh(summ.index, summ[col], left=acc, color=colr, label=col, height=0.65); acc += summ[col].values
ax.set_title('Integrity checks per table: how many passed, were informational, or need a decision'); ax.set_xlabel('number of checks'); ax.legend(ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.14)); ax.grid(axis='y', visible=False)
save(fig, '06b_integrity_checks')

**How to read this:** `PASS` = the check found nothing. `INFO` = something was flagged but it is expected/harmless (explained in `note`). `WARN` = flagged and needs a decision. The next cell inspects the text labels we will rely on later (country ↔ nationality for Question 2, status labels for Question 3).

In [ ]:
print('circuits.country      :', ci.country.nunique(), 'values  →', sorted(ci.country.unique()))
print('\ndrivers.nationality   :', dv.nationality.nunique(), 'values  →', sorted(dv.nationality.unique()))
print('\nconstructors.nationality:', co.nationality.nunique(), 'values  →', sorted(co.nationality.unique()))
print(f"\nstatus labels: {st.status.nunique()} distinct, e.g.", st.status.sample(12, random_state=1).tolist())

### 3.6 Save the cleaned artefacts
We write **all 14 cleaned tables** (same file names as the original dataset, but with real types, `\\N`/empty → missing, duplicates removed in `results`) plus the modelling table.

In [ ]:
MODEL_BASE.to_csv(PROC_DIR / 'results_clean.csv', index=False)
DROPPED.to_csv(PROC_DIR / 'duplicates_removed.csv', index=False)
pd.DataFrame(LOG).to_csv(PROC_DIR / 'cleaning_log.csv', index=False)
# ---- the full cleaned set: same 14 file names as the original dataset, but cleaned & typed ------------------
TABLE_DIR = PROC_DIR / 'clean_tables'; TABLE_DIR.mkdir(exist_ok=True)
def _hms(x): return '' if pd.isna(x) else str(x).split(' days ')[-1][:8]      # timedelta -> 'HH:MM:SS'
out_tables = {t: CLEAN[t].copy() for t in TABLES}
out_tables['results'] = R.drop(columns=['no_grid_slot']).copy()               # de-duplicated, ALL rows, + 'podium' and 'started' flags
for t, df in out_tables.items():
    for c in df.columns:
        if pd.api.types.is_timedelta64_dtype(df[c]): df[c] = df[c].map(_hms)
    df.to_csv(TABLE_DIR / f'{t}.csv', index=False)
rows = [(t, len(out_tables[t]), out_tables[t].shape[1]) for t in TABLES]
print(f'saved {len(TABLES)} cleaned tables to', TABLE_DIR.resolve())
print('results.csv here = de-duplicated, all rows (incl. never-started, flagged by `started`); results_clean.csv = modelling base (started only)')
print('saved to', PROC_DIR.resolve()); print(*sorted(p.name for p in PROC_DIR.glob('*.csv')), sep='\n  ')
pd.DataFrame(rows, columns=['table', 'rows', 'cols']).set_index('table')

## 4 · Re-EDA — EDA **after** cleaning (what did cleaning change?)
Each plot puts **before (blue)** next to **after (orange)** so the impact of every step is visible, as the brief asks.

### 4.1 Row counts, sentinels and typing – before vs after

In [ ]:
summary = pd.DataFrame({
    'rows raw': {t: len(RAW[t]) for t in TABLES}, 'rows clean': {t: len(CLEAN[t]) for t in TABLES},
    'sentinels raw': {t: int(SENT[t].sum()) for t in TABLES}, 'sentinels clean': left,
    'NaN in original cols (clean)': {t: int(CLEAN[t][[c for c in RAW[t].columns if c in CLEAN[t].columns]].isna().sum().sum()) for t in TABLES}})
summary.loc['results → modelling base'] = [len(RAW['results']), len(MODEL_BASE), int(SENT['results'].sum()), 0, int(MODEL_BASE[RAW['results'].columns].isna().sum().sum())]
print(summary.to_string())
print('(constructor_results: the sparse `status` column was replaced by a boolean flag, so it no longer counts as NaN)')

# columns whose dtype was "object" in a naive load but became numeric/date/time after typing
rows = []
for t in TABLES:
    for c, dt in NAIVE_DTYPES[t].items():
        if c not in CLEAN[t].columns: continue          # e.g. constructor_results.status became a boolean flag
        new = CLEAN[t][c].dtype
        if (dt == object or pd.api.types.is_string_dtype(dt)) and (pd.api.types.is_numeric_dtype(new) or pd.api.types.is_datetime64_any_dtype(new) or pd.api.types.is_timedelta64_dtype(new)):
            rows.append((t, c, str(dt), str(new)))
retyped = pd.DataFrame(rows, columns=['table', 'column', 'naive dtype', 'clean dtype'])
newcols = {t: [c for c in CLEAN[t].columns if c not in RAW[t].columns] for t in TABLES}
print(f'\ncolumns that were text in a naive load and are now numeric/date/time: {len(retyped)}')
print('new derived columns:', {t: v for t, v in newcols.items() if v})
fig, ax = plt.subplots(figsize=(7.5, 3.6))
cnt = retyped.groupby('table').size().sort_values()
ax.barh(cnt.index, cnt.values, color=ORANGE, height=0.6)
for y, v in enumerate(cnt.values): ax.text(v + 0.1, y, str(v), va='center', fontsize=9)
ax.set_title('Columns re-typed from text (naive load) to numeric / date / time'); ax.grid(axis='y', visible=False)
save(fig, '07_retyped_columns')
retyped

**Reading it:** a naïve load makes every column that contains a single `\N` a *text* column – e.g. `results.position`, `races.date`, `drivers.dob`. After cleaning they are proper integers/dates/times (chart above). Duration strings such as `qualifying.q1` are kept as the original text **and** get a numeric sibling (`q1_s`, `time_s`, `duration_s`… listed under “new derived columns”), so nothing is lost. **0** sentinel strings remain. Row counts are unchanged everywhere **except** `results`, which loses exactly the duplicate rows (§3.3) and the never-started entries (§3.4).

### 4.2 Impact on the target and the field – raw vs cleaned

In [ ]:
raw_r = res.copy()                                   # raw results joined with year (from §2.5)
cl = MODEL_BASE.copy(); cl['era'] = era_of(cl.year)
def era_stats(d):
    g = d.groupby('era', observed=True)
    out = pd.DataFrame({'podium_rate': g.podium.mean(), 'entrants_per_race': g.size() / g.raceId.nunique()})
    return out
a, b = era_stats(raw_r), era_stats(cl)
tab = a.join(b, lsuffix='_raw', rsuffix='_clean'); print(tab.to_string())
print(f'\noverall podium rate: raw {raw_r.podium.mean():.2%} → clean {cl.podium.mean():.2%} | rows {len(raw_r):,} → {len(cl):,}')

x = np.arange(len(a)); w = 0.38
fig, axes = plt.subplots(1, 2, figsize=(12, 3.9))
for ax, col, ttl, fmt_ in [(axes[0], 'podium_rate', 'Podium rate by era', True), (axes[1], 'entrants_per_race', 'Entrants per race by era', False)]:
    ax.bar(x - w/2, a[col], w, color=BLUE, label='raw'); ax.bar(x + w/2, b[col], w, color=ORANGE, label='cleaned')
    ax.set_xticks(x); ax.set_xticklabels(a.index, fontsize=8); ax.set_title(ttl); ax.grid(axis='x', visible=False)
    ax.set_ylim(0, max(a[col].max(), b[col].max()) * 1.28); ax.legend(ncol=2, loc='upper left')
    if fmt_: ax.yaxis.set_major_formatter(PercentFormatter(1.0))
save(fig, '08_before_after_target_by_era')

**Reading it:** removing never-started entries raises the podium rate (the denominator now contains only drivers who actually raced) and makes the field size comparable across eras. The effect is largest in the early/pre-1995 eras, where many entries never qualified. The modern-era numbers barely change – confirming the cleaning mostly removes *historical* noise and does not distort the era we are most likely to model.

### 4.3 Parsed durations – distributions (does the conversion look sane?)
Also a first look at **why raw qualifying seconds are not a good feature as-is**: a lap at Monaco (~72 s) vs Spa (~105 s) is not comparable → we will normalise each time relative to the session’s best in Part 2.

In [ ]:
qq = CLEAN['qualifying'].merge(CLEAN['races'][['raceId', 'year']], on='raceId')
qq = qq[qq.year >= 2006].copy()
qq['best_s'] = qq[['q1_s', 'q2_s', 'q3_s']].min(axis=1)
qq['rel_to_pole'] = qq.best_s / qq.groupby('raceId').best_s.transform('min')
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].hist(qq.q1_s.dropna(), bins=60, color=ORANGE); axes[0].set_title('q1 in seconds (2006+)\nmixes tracks – not comparable'); axes[0].set_xlabel('seconds')
axes[1].hist(qq.rel_to_pole.dropna().clip(upper=1.15), bins=60, color=AQUA); axes[1].set_title('best qualifying time ÷ pole time\ncomparable across tracks'); axes[1].set_xlabel('ratio (1.00 = pole; values > 1.15 pooled in last bin)')
axes[2].hist(ps.duration_s[ps.duration_s < 45].dropna(), bins=60, color=BLUE); axes[2].set_title('pit-stop duration (s, < 45 s)\nright-skewed with a long tail'); axes[2].set_xlabel('seconds')
for a_ in axes: a_.set_ylabel('count'); a_.grid(axis='x', visible=False)
save(fig, '09_parsed_duration_distributions')

**Reading it:** the converted times are plausible (qualifying laps cluster between ~65–110 s; pit stops mostly 20–30 s with a long right tail). The middle panel shows the same laps after dividing by the pole time: a right-skewed distribution starting at 1.00 (pole itself) that is comparable across circuits and years – the kind of transformation Part 2 will use and justify.

### 4.4 Qualifying coverage on the cleaned table + candidate modelling windows
The brief’s split (chronological, from the primer): **train ≤ 2019 · validation 2020–2021 · test ≥ 2022**. We check how much data each candidate training window gives, and how complete qualifying is in it, so the window choice in Part 2 is evidence-based.

In [ ]:
C = MODEL_BASE.merge(CLEAN['qualifying'][['raceId', 'driverId', 'position', 'q1_s']].rename(columns={'position': 'q_position'}),
                     on=['raceId', 'driverId'], how='left')
C['has_quali'] = C.q_position.notna()
C['split'] = np.select([C.year <= 2019, C.year <= 2021], ['train', 'val'], 'test')
rows = []
for start in [1950, 1995, 2000, 2005, 2006, 2010, 2014]:
    d_ = C[C.year >= start]
    tr = d_[d_.split == 'train']
    rows.append(dict(train_start=start, train_rows=len(tr), train_races=tr.raceId.nunique(), train_podium_rate=tr.podium.mean(),
                     train_quali_coverage=tr.has_quali.mean(), val_rows=int((d_.split == 'val').sum()), test_rows=int((d_.split == 'test').sum()),
                     test_podium_rate=d_[d_.split == 'test'].podium.mean()))
win = pd.DataFrame(rows); print(win.to_string(index=False))
print('\nqualifying coverage by split (full population):'); print(C.groupby('split').has_quali.mean().round(3).to_string())

cv = C.groupby('year').has_quali.mean()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(cv.index, cv.values, color=ORANGE, lw=2); axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[0].axvspan(1949.5, 2019.5, color=BLUE, alpha=0.06); axes[0].axvspan(2019.5, 2021.5, color=AQUA, alpha=0.12); axes[0].axvspan(2021.5, 2024.5, color=MAGENTA, alpha=0.12)
axes[0].text(1972, 0.05, 'train ≤ 2019', ha='center', fontsize=8, color=INK2)
for xx, tt in [(2020.5, 'val'), (2023, 'test')]: axes[0].text(xx, 0.5, tt, ha='center', va='center', rotation=90, fontsize=8, color=INK2)
axes[0].set_title('Qualifying coverage of the cleaned population, with the split'); axes[0].set_xlabel('season')
axes[1].plot(win.train_start.astype(str), win.train_quali_coverage, 'o-', color=ORANGE, lw=2, label='qualifying coverage in train')
axes[1].yaxis.set_major_formatter(PercentFormatter(1.0)); axes[1].set_ylim(0, 1.05); axes[1].set_xlabel('first season of the training window'); axes[1].legend(loc='lower right')
axes[1].set_title('Restricting the training window → completeness')
save(fig, '10_modelling_window_candidates')

**Reading it & decision for Part 2 (hypothesis from EDA, to be tested empirically):** shrinking the training window to the seasons with complete qualifying data trades **rows** for **feature quality**. The table above quantifies both sides (rows, podium rate, qualifying coverage). In Part 2 we will test windows empirically (before/after on the same pipeline) rather than assume. The per-split coverage printed above shows whether validation (2020–21) and test (≥2022) are complete, so the evaluation sets are clean whichever training window we choose.

## 5 · Decision log & hand-off

In [ ]:
# raw files must be byte-identical to when we loaded them
HASH_AFTER = {t: md5(RAW_DIR / f'{t}.csv') for t in TABLES}
assert HASH_AFTER == HASH_BEFORE, 'a raw file changed!'
print('✔ all', len(TABLES), 'raw CSV files are byte-identical to the start of the notebook (md5 verified)')
print(f"✔ MODEL_BASE: {len(MODEL_BASE):,} rows, one per (raceId, driverId): {not MODEL_BASE.duplicated(KEY).any()}  |  podium rate {MODEL_BASE.podium.mean():.2%}")
print(f"✔ seasons {int(MODEL_BASE.year.min())}–{int(MODEL_BASE.year.max())}, {MODEL_BASE.raceId.nunique():,} races, {MODEL_BASE.driverId.nunique():,} drivers")

| # | Decision | Evidence (section) | Alternative we measured/considered |
|---|---|---|---|
| 1 | Treat only literal `\N` **and the empty string** as sentinels (`keep_default_na=False`) | §1, §2.2 (other-token scan) | pandas default NA list (would also eat `"NA"` strings) |
| 2 | Keep structural missingness (q2/q3, fastest lap) as `NaN`, no blind imputation | §2.2, §2.8 | mean-fill (would erase “knocked out in Q1”) |
| 3 | One duration parser for all layouts; verified against millisecond columns | §2.9, §3.2 | `float()` per column (crashes on `m:ss.mmm`) |
| 4 | Duplicates → `best_position` rule (official shared-drive credit) | §2.5, §3.3 | `most_laps`, `first_result` (compared in §3.3) |
| 5 | Exclude entries with no grid slot; keep pit-lane starts | §2.7, §3.4 | keep everything (`CFG['exclude_non_starters']=False`) |
| 6 | Treat `grid = 0` as *not a position* (pit lane / no start) | §2.7 | using 0 as “better than pole” |
| 7 | Chronological split train ≤2019 / val 2020–21 / test ≥2022 (from the primer) | §4.4 | random split (would leak era / form information) |
| 8 | Leakage map fixed before feature work | §2.10 | – |
| 9 | Train from 2003 (complete qualifying data) + a `knockout_format` flag | §9.1, §10.2 | 1950 / 1994 / 2006 starts (no clear difference) |
| 10 | 22 final features; `quali_pos` kept next to `grid_pos` | §9.6–§9.7, §11.1 | dropping weak / duplicate features |
| 11 | Yes/no from a per-model threshold chosen on validation (best F1) | §10.4 | fixed 0.5; top-3 per race |
| 12 | All four models reported on the test years; logistic regression used for the worked explanations (simplest model within noise of the best) | §10.3, §12, §13 | picking the single best model on test scores |

**Known limitations so far:** (a) shared-drive rows (≤1978) may carry the car’s grid slot; (b) qualifying coverage drifts before 2003 and the knockout format only exists from 2006; (c) the non-starter filter reconstructs a pre-race fact from post-race columns (never used as features), and a pit-lane starter who retired on lap 1 looks like a non-starter and is excluded (a handful of rows); (d) thresholds chosen on 2020–21 do not transfer perfectly to 2022–24 (§12); (e) explanations (§13) describe what the models rely on, not what causes podiums.

### ▶ Next: Part 2
Q1–Q3 are answered in §6–§8, features and the split in §9, models in §10, experiments in §11, the test in §12, explanations in §13 and the inference function in §14.

---
# Part 2 · Data-engineering questions

## 6 · Q1 — Which circuits convert a front-row start into a podium most often? Does the ranking change after 2014?

* **Conversion rate** = podiums ÷ front-row starts (for one circuit).
* **Front row** = grid position 1 or 2, i.e. where the car actually *starts*. Qualifying position can differ after penalties, so we keep it separate (checked in 6.1).
* **Two eras:** 2003–2013 and 2014–2024 (11 seasons each, qualifying data complete in both).
* **Multi-hop join:** `results` → `races` (year, circuit) → `circuits` (name), and `results` + `qualifying` on (`raceId`, `driverId`).

In [ ]:
Q1_PRE, Q1_POST, MIN_N = (2003, 2013), (2014, 2024), 10     # the two eras; minimum front-row starts per circuit in each era

qual = CLEAN['qualifying'][['raceId', 'driverId', 'position']].rename(columns={'position': 'quali_pos'})
assert not qual.duplicated(['raceId', 'driverId']).any()
Q1 = (MODEL_BASE[['raceId', 'driverId', 'year', 'circuitId', 'grid', 'podium']]
      .merge(qual, on=['raceId', 'driverId'], how='left', validate='one_to_one')
      .merge(CLEAN['circuits'][['circuitId', 'circuitRef']], on='circuitId', how='left', validate='many_to_one'))
Q1['quali_pos'] = pd.to_numeric(Q1['quali_pos'])
Q1['grid_fr']  = Q1.grid.between(1, 2)          # front-row START
Q1['quali_fr'] = Q1.quali_pos.between(1, 2)     # front-row in QUALIFYING

assert not Q1.duplicated(KEY).any() and len(Q1) == len(MODEL_BASE)
print(f'grain check: {len(Q1):,} rows, one per (raceId, driverId), joins added no rows')
w = pd.cut(Q1.year, [1949, 1995, 2002, 2013, 2024], labels=['1950–95', '1996–2002', '2003–13', '2014–24'])
print('\nshare of drivers with a qualifying position:'); print(Q1.groupby(w, observed=True).quali_pos.apply(lambda s: s.notna().mean()).round(3).to_string())

### 6.1 Grid vs qualifying position
Do the two definitions of "front row" differ, and does it matter?

In [ ]:
W = Q1[Q1.year.between(Q1_PRE[0], Q1_POST[1])]
B = W[W.quali_fr | W.grid_fr].copy()
B['group'] = np.select([B.quali_fr & B.grid_fr, B.quali_fr & ~B.grid_fr], ['qualified front row\nand started there', 'qualified front row\nbut started further back'], default='qualified lower\nbut started front row')
g = B.groupby('group').podium.agg(['size', 'mean']).rename(columns={'size': 'drivers', 'mean': 'podium rate'})
order = ['qualified front row\nand started there', 'qualified front row\nbut started further back', 'qualified lower\nbut started front row']
g = g.loc[order]; print(g.round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(range(3), g['podium rate'], color=[BLUE, MAGENTA, AQUA], width=0.55, zorder=3)
for i, (c, n) in enumerate(zip(g['podium rate'], g.drivers)): ax.text(i, c + 0.02, f'{c:.0%}\n(n={n})', ha='center', va='bottom', fontsize=9)
ax.set_xticks(range(3)); ax.set_xticklabels(order, fontsize=8.5); ax.set_ylim(0, 1.0); ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylabel('finished on the podium'); ax.set_title('The starting slot matters more than the qualifying result')
save(fig, '11_q1_grid_vs_qualifying')

Only 4 % of front-row qualifiers started further back, but when they did, they converted far less (31 % vs 73 %). So **grid** is the right definition of a front-row start.

### 6.2 Ranking per era
Only circuits with at least 10 front-row starts in **both** eras are ranked (fewer starts = unreliable rate).

In [ ]:
from scipy import stats

def era_table(df, flag, pre=Q1_PRE, post=Q1_POST, min_n=MIN_N):
    f = df[df[flag]]
    a = f[f.year.between(*pre)].groupby('circuitRef').podium.agg(['sum', 'size']).add_prefix('pre_')
    b = f[f.year.between(*post)].groupby('circuitRef').podium.agg(['sum', 'size']).add_prefix('post_')
    t = a.join(b, how='outer').fillna(0).astype(int)
    t['pre_rate'] = t.pre_sum / t.pre_size.replace(0, np.nan); t['post_rate'] = t.post_sum / t.post_size.replace(0, np.nan)
    t['ranked'] = (t.pre_size >= min_n) & (t.post_size >= min_n)
    return t

T = era_table(Q1, 'grid_fr')
rk = T[T.ranked].copy()
rk['pre_rank']  = rk.pre_rate.rank(ascending=False, method='min').astype(int)
rk['post_rank'] = rk.post_rate.rank(ascending=False, method='min').astype(int)
rk['rank_change'] = rk.pre_rank - rk.post_rank        # + = moved up after 2014
print(f'{len(rk)} circuits ranked\n')
print(rk.sort_values('post_rate', ascending=False)[['pre_size', 'pre_rate', 'pre_rank', 'post_size', 'post_rate', 'post_rank', 'rank_change']].round(2).to_string())
print(f'\nnot ranked (too few starts in one era): {", ".join(T[~T.ranked].index)}')
print(f"\noverall conversion  {Q1_PRE[0]}–{Q1_PRE[1]}: {rk.pre_sum.sum() / rk.pre_size.sum():.0%}    {Q1_POST[0]}–{Q1_POST[1]}: {rk.post_sum.sum() / rk.post_size.sum():.0%}")
rho = stats.spearmanr(rk.pre_rate, rk.post_rate)[0]
print(f'rank correlation between the two eras: {rho:.2f}   (1 = same order, 0 = unrelated)')

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), gridspec_kw={'width_ratios': [1.2, 1]})
ax = axes[0]; d = rk.sort_values('post_rate'); y = np.arange(len(d))
ax.hlines(y, d.pre_rate, d.post_rate, color='#bdbdb8', lw=2, zorder=2)
ax.scatter(d.pre_rate, y, color=BLUE, s=40, zorder=3, label=f'{Q1_PRE[0]}–{Q1_PRE[1]}')
ax.scatter(d.post_rate, y, color=ORANGE, s=40, zorder=3, label=f'{Q1_POST[0]}–{Q1_POST[1]}')
ax.set_yticks(y); ax.set_yticklabels([f"{n.replace('_', ' ').title()}  ({a_}|{b_})" for n, a_, b_ in zip(d.index, d.pre_size, d.post_size)], fontsize=8.5)
ax.set_xlim(0.45, 0.95); ax.xaxis.set_major_formatter(PercentFormatter(1.0)); ax.legend(loc='upper left')
ax.set_xlabel('front-row starts that ended on the podium     (numbers in brackets = front-row starts before | after)')
ax.set_title('Front-row → podium conversion by circuit')
ax = axes[1]
ax.plot([0.45, 1], [0.45, 1], color=INK2, lw=1, ls='--', zorder=1)
ax.text(0.83, 0.855, 'same rate in both eras', ha='center', va='bottom', fontsize=8, color=INK2, rotation=42, rotation_mode='anchor')
ax.scatter(rk.pre_rate, rk.post_rate, color=ORANGE, s=40, zorder=3)
OFF = {'catalunya': (-5, 3, 'right'), 'interlagos': (-5, 3, 'right'), 'villeneuve': (5, -9, 'left'), 'marina_bay': (-5, -9, 'right'), 'suzuka': (-5, 3, 'right')}
for nm, r in rk.iterrows():
    dx, dy, ha_ = OFF.get(nm, (5, 3, 'left')); ax.annotate(nm.replace('_', ' ').title(), (r.pre_rate, r.post_rate), fontsize=7.5, color=INK2, xytext=(dx, dy), ha=ha_, textcoords='offset points')
ax.set_xlim(0.45, 1.0); ax.set_ylim(0.45, 1.0); ax.xaxis.set_major_formatter(PercentFormatter(1.0)); ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel(f'{Q1_PRE[0]}–{Q1_PRE[1]}'); ax.set_ylabel(f'{Q1_POST[0]}–{Q1_POST[1]}'); ax.set_title(f'Did the order change?  rank correlation = {rho:.2f}')
save(fig, '12_q1_conversion_by_circuit')

### 6.3 Check: other definition and other starting years

In [ ]:
rows = []
for flag, label, pres in [('grid_fr', 'grid ≤ 2', [(2003, 2013), (1996, 2013), (1950, 2013)]), ('quali_fr', 'qualifying position ≤ 2', [(2003, 2013)])]:
    for pre in pres:
        t = era_table(Q1, flag, pre=pre); t = t[t.ranked]
        rows.append(dict(definition=label, before_era=f'{pre[0]}–{pre[1]}', circuits=len(t), rank_correlation=round(stats.spearmanr(t.pre_rate, t.post_rate)[0], 2),
                         conversion_before=round(t.pre_sum.sum() / t.pre_size.sum(), 2), conversion_after=round(t.post_sum.sum() / t.post_size.sum(), 2)))
print(pd.DataFrame(rows).to_string(index=False))

### 6.4 Answer
* **Highest conversion (2014–24):** Monaco and Suzuka (89 %), Yas Marina (86 %). **Lowest:** Hungaroring (59 %), Bahrain (67 %).
* **After 2014:** conversion went up almost everywhere (**67 % → 77 %**), but the *order* of circuits changed only weakly (rank correlation 0.36; about −0.1 with older starting years). Each circuit has only ~20 front-row starts per era, so single moves (e.g. Monza −9 places) should not be trusted. What holds in both eras: Monaco/Suzuka high, Hungaroring low.
* **Grain:** one row per `(raceId, driverId)`; the front row is 2 starters per race.
* **Missingness:** qualifying position exists for 99.8 % of drivers from 2003, but only 29 % in 1996–2002 and 6 % before; 24 circuits are not ranked (< 10 starts in one era) and are listed above.
* **Pre-race cutoff:** grid and qualifying are known before the race, so both are allowed features. The podium is the outcome, so this analysis is descriptive; a "circuit conversion" feature must later use **earlier races only**.

---
## 7 · Q2 — Do drivers podium more often at a circuit in their home country, after controlling for starting position?

* **Home race** = the driver's nationality (`drivers`) matches the circuit's country (`circuits`). The two files spell these differently ("British" vs "UK"), so we write a small mapping table by hand.
* **Controlling for starting position** = compare home and away drivers who started from **similar grid slots** (grid groups: 1, 2–3, 4–6, 7–10, 11–15, 16+).
* **Multi-hop join:** `results` → `races` (circuit, name) → `circuits` (country), and `results` → `drivers` (nationality).
* **Left out:** the *Indianapolis 500* (1950–60, counted as a championship race but almost no F1 drivers entered; it would distort "American home races"), and **grid = 0** (pit-lane start, not a grid slot).

In [ ]:
# nationality (drivers.csv) → country of the circuit (circuits.csv). Dual nationalities count as home in both countries.
NAT_TO_COUNTRY = {
    'American': ['USA'], 'American-Italian': ['USA', 'Italy'], 'Argentine': ['Argentina'], 'Argentine-Italian': ['Argentina', 'Italy'],
    'Australian': ['Australia'], 'Austrian': ['Austria'], 'Belgian': ['Belgium'], 'Brazilian': ['Brazil'], 'British': ['UK'],
    'Canadian': ['Canada'], 'Chinese': ['China'], 'Dutch': ['Netherlands'], 'French': ['France'], 'German': ['Germany'], 'East German': ['Germany'],
    'Hungarian': ['Hungary'], 'Indian': ['India'], 'Italian': ['Italy'], 'Japanese': ['Japan'], 'Malaysian': ['Malaysia'],
    'Mexican': ['Mexico'], 'Monegasque': ['Monaco'], 'Portuguese': ['Portugal'], 'Russian': ['Russia'], 'South African': ['South Africa'],
    'Spanish': ['Spain'], 'Swedish': ['Sweden'], 'Swiss': ['Switzerland']}
assert all(c in set(CLEAN['circuits'].country) for v in NAT_TO_COUNTRY.values() for c in v), 'mapping points to a country with no circuit'

Q2 = (MODEL_BASE[['raceId', 'driverId', 'year', 'circuitId', 'grid', 'podium']]
      .merge(CLEAN['races'][['raceId', 'name']].rename(columns={'name': 'gp'}), on='raceId', how='left', validate='many_to_one')
      .merge(CLEAN['circuits'][['circuitId', 'country']], on='circuitId', how='left', validate='many_to_one')
      .merge(CLEAN['drivers'][['driverId', 'nationality']], on='driverId', how='left', validate='many_to_one'))
assert len(Q2) == len(MODEL_BASE) and not Q2.duplicated(KEY).any() and Q2.nationality.notna().all()
n0 = len(Q2)
Q2 = Q2[(Q2.gp != 'Indianapolis 500') & (Q2.grid > 0)].copy()
Q2['home'] = [c in NAT_TO_COUNTRY.get(n, []) for n, c in zip(Q2.nationality, Q2.country)]
Q2['grid_group'] = pd.cut(Q2.grid, [0, 1, 3, 6, 10, 15, 99], labels=['1', '2–3', '4–6', '7–10', '11–15', '16+'])

print(f'grain check: one row per (raceId, driverId); no missing nationality')
print(f'rows {n0:,} → {len(Q2):,} after removing Indianapolis 500 and grid = 0')
print(f'home starts: {int(Q2.home.sum()):,} ({Q2.home.mean():.1%} of all starts)')
print(f'nationalities with no home circuit in the data (never "home"): {", ".join(sorted(set(CLEAN["drivers"].nationality) - set(NAT_TO_COUNTRY)))}')
print(f'\nraw podium rate  home: {Q2[Q2.home].podium.mean():.1%}   away: {Q2[~Q2.home].podium.mean():.1%}   (before controlling for grid)')

### 7.1 Same grid group, home vs away

In [ ]:
def by_grid(df):
    t = df.groupby(['grid_group', 'home'], observed=True).podium.agg(['mean', 'size']).unstack()
    t.columns = ['away_rate', 'home_rate', 'away_starts', 'home_starts']
    return t

def expected_vs_actual(df):
    t = by_grid(df)
    exp = (t.away_rate * t.home_starts).sum()                                # home podiums if home drivers did as well as away drivers from the same grid group
    act = (t.home_rate * t.home_starts).sum()
    luck = np.sqrt((t.home_starts * t.away_rate * (1 - t.away_rate)).sum())  # typical swing from luck alone
    return exp, act, luck

T2 = by_grid(Q2); print(T2.round(3).to_string())
res_q2 = {'1950–2024': expected_vs_actual(Q2), '2003–2024': expected_vs_actual(Q2[Q2.year >= 2003])}
for k, (e, a, l) in res_q2.items(): print(f'\n{k}: home podiums expected from grid = {e:.0f}, actual = {a:.0f}   (luck alone moves it by about ±{l:.0f})')

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), gridspec_kw={'width_ratios': [1.5, 1]})
ax = axes[0]; x = np.arange(len(T2)); wd = 0.38
ax.bar(x - wd / 2, T2.away_rate, wd, color=BLUE, label='away drivers', zorder=3); ax.bar(x + wd / 2, T2.home_rate, wd, color=ORANGE, label='home drivers', zorder=3)
ax.set_xticks(x); ax.set_xticklabels([f'{g}\n({a:,} | {h})' for g, a, h in zip(T2.index, T2.away_starts, T2.home_starts)], fontsize=8.5); ax.yaxis.set_major_formatter(PercentFormatter(1.0)); ax.legend(loc='upper right')
ax.set_xlabel('starting grid position   (brackets = away starts | home starts)'); ax.set_ylabel('finished on the podium'); ax.set_title('Same grid group: home vs away')
ax = axes[1]; x = np.arange(2)
ex = [v[0] for v in res_q2.values()]; ac = [v[1] for v in res_q2.values()]
ax.bar(x - wd / 2, ex, wd, color=BLUE, label='expected from grid', zorder=3); ax.bar(x + wd / 2, ac, wd, color=ORANGE, label='actual', zorder=3)
for i in range(2): ax.text(i - wd / 2, ex[i] + 4, f'{ex[i]:.0f}', ha='center', fontsize=9); ax.text(i + wd / 2, ac[i] + 4, f'{ac[i]:.0f}', ha='center', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(list(res_q2)); ax.set_ylim(0, max(ex) * 1.2); ax.legend(loc='upper right'); ax.set_ylabel('home-driver podiums'); ax.set_title('Home podiums: expected vs actual')
save(fig, '13_q2_home_vs_away')

### 7.2 Answer
* **No.** Raw, home drivers podium *less* often (12.0 % vs 13.9 %), but they also start further back on average, so that comparison is unfair. Once we compare drivers from the **same grid group**, the two are almost identical.
* **In numbers:** home drivers got **230** podiums; drivers with the same grid slots would be expected to get **236**. Luck alone moves this by about ±12, so the gap is not meaningful. The modern era (2003–24) again shows no advantage: slightly *fewer* podiums than expected (52 vs 59, luck about ±5).
* **Grain:** one row per `(raceId, driverId)`; "home" is decided per row.
* **Missingness:** no driver has a missing nationality; 15 nationalities (e.g. Finnish, Polish, Thai) never have a home circuit and are always "away". The Indianapolis 500 and pit-lane starts are removed, and the mapping is written by hand and documented above.
* **Pre-race cutoff:** nationality, circuit country and grid are all known before the race, so "home race" is an allowed feature for Part 2. The podium is the outcome. Limit: we controlled for grid only, not for car quality.

---
## 8 · Q3 — Which constructors have the highest mechanical-retirement rate in 2014–2021 vs 2022–2024?

* **Mechanical retirement** = a race start that ended early because a **car component failed** (engine, gearbox, brakes…). Not counted: crashes, driver errors, disqualifications.
* **Rate** = mechanical retirements ÷ race starts, per constructor and era.
* **Multi-hop join:** `results` (status id) → `status` (reason text) and `results` → `constructors` (team name).
* **The grouping is ours**: the data has 100+ status texts, so we sort every status seen in 2014–2024 into groups and document it below.

In [ ]:
Q3_ERAS = {'2014–2021': (2014, 2021), '2022–2024': (2022, 2024)}
MIN_STARTS = 40                       # a team needs at least this many race starts in an era to be ranked

MECHANICAL = ['Engine', 'Brakes', 'Gearbox', 'Power Unit', 'Suspension', 'Electrical', 'Hydraulics', 'Power loss', 'Oil leak', 'Overheating',
              'Mechanical', 'Water pressure', 'Turbo', 'Fuel pressure', 'Water leak', 'Electronics', 'ERS', 'Exhaust', 'Battery', 'Transmission',
              'Radiator', 'Throttle', 'Technical', 'Vibrations', 'Clutch', 'Oil pressure', 'Steering', 'Fuel pump', 'Fuel leak', 'Driveshaft',
              'Differential', 'Cooling system', 'Brake duct', 'Drivetrain', 'Spark plugs', 'Water pump',
              'Wheel nut', 'Seat']                       # team decision: counted as car failures
BORDERLINE = ['Retired', 'Out of fuel', 'Debris']          # cause not clear from the name (no reason given / fuel miscalculation / object on track)
NOT_MECHANICAL = ['Collision', 'Accident', 'Collision damage', 'Spun off', 'Disqualified', 'Excluded', 'Withdrew', 'Illness',      # crash / driver / regulation
                  'Wheel', 'Puncture', 'Tyre', 'Undertray', 'Rear wing', 'Front wing', 'Damage']                                 # team decision: could be driver error or contact

Q3 = (MODEL_BASE[MODEL_BASE.year.between(2014, 2024)][['raceId', 'driverId', 'constructorId', 'year', 'status']]
      .merge(CLEAN['constructors'][['constructorId', 'name']].rename(columns={'name': 'team'}), on='constructorId', how='left', validate='many_to_one'))
assert not Q3.duplicated(KEY).any() and Q3.status.notna().all() and Q3.team.notna().all()
Q3['reached_flag'] = Q3.status.eq('Finished') | Q3.status.str.startswith('+')          # Finished, +1 Lap, +2 Laps ...
known = set(MECHANICAL) | set(BORDERLINE) | set(NOT_MECHANICAL)
assert (Q3.reached_flag | Q3.status.isin(known)).all(), sorted(set(Q3.status[~(Q3.reached_flag | Q3.status.isin(known))]))
Q3['mech'] = Q3.status.isin(MECHANICAL)
Q3['mech_broad'] = Q3.status.isin(MECHANICAL + BORDERLINE)       # sensitivity check only
Q3['era'] = np.where(Q3.year <= 2021, '2014–2021', '2022–2024')

grp = Q3.assign(group=np.select([Q3.reached_flag, Q3.mech, Q3.status.isin(BORDERLINE)], ['reached the flag', 'MECHANICAL (counted)', 'borderline (not counted)'], 'crash / driver / regulation'))
print(f'grain check: {len(Q3):,} race starts, one per (raceId, driverId); every status text is classified (none left over)\n')
print(grp.groupby('group').status.agg(starts='size', different_status_texts='nunique').to_string())
print('\nmost common statuses per group:')
for gname, gdf in grp[~grp.reached_flag].groupby('group'): print(f'  {gname}: ' + ', '.join(f'{k} ({v})' for k, v in gdf.status.value_counts().head(8).items()))

### 8.1 Rate per constructor

In [ ]:
t3 = Q3.groupby(['era', 'team']).agg(starts=('mech', 'size'), mech_retirements=('mech', 'sum'), broad=('mech_broad', 'sum')).reset_index()
t3['rate'] = t3.mech_retirements / t3.starts; t3['rate_broad'] = t3.broad / t3.starts
shown = t3[t3.starts >= MIN_STARTS]
for era in Q3_ERAS:
    e = shown[shown.era == era].sort_values('rate', ascending=False)
    print(f'{era}  (all teams together: {Q3[Q3.era == era].mech.mean():.1%} of starts)'); print(e[['team', 'starts', 'mech_retirements', 'rate']].round(3).to_string(index=False))
    print(f'  not ranked (< {MIN_STARTS} starts): ' + ', '.join(f'{r.team} ({r.starts})' for r in t3[(t3.era == era) & (t3.starts < MIN_STARTS)].itertuples()) + '\n')

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), sharex=True)
for ax, (era, colour) in zip(axes, [('2014–2021', BLUE), ('2022–2024', ORANGE)]):
    e = shown[shown.era == era].sort_values('rate'); y = np.arange(len(e))
    ax.barh(y, e.rate, color=colour, height=0.65, zorder=3)
    for yi, r in zip(y, e.itertuples()): ax.text(r.rate + 0.003, yi, f'{r.rate:.1%}', va='center', fontsize=8.5, zorder=5, bbox=dict(facecolor='white', edgecolor='none', pad=0.6))
    ax.set_yticks(y); ax.set_yticklabels([f'{r.team}  ({r.starts})' for r in e.itertuples()], fontsize=8.5)
    ax.axvline(Q3[Q3.era == era].mech.mean(), color=INK2, ls='--', lw=1, zorder=2)
    ax.set_title(f'{era}   (dashed line = all teams: {Q3[Q3.era == era].mech.mean():.1%})'); ax.xaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_xlabel('race starts ending in a mechanical failure   (brackets = race starts)'); ax.set_xlim(0, 0.2)
save(fig, '14_q3_mechanical_retirements')

### 8.2 Check: what if the 3 borderline statuses (Retired, Out of fuel, Debris) were counted too?

In [ ]:
chk = []
for era in Q3_ERAS:
    e = t3[(t3.era == era) & (t3.starts >= MIN_STARTS)]
    top = lambda col: ', '.join(e.sort_values(col, ascending=False).team.head(3))
    chk.append(dict(era=era, top3_main=top('rate'), top3_if_borderline_counted=top('rate_broad'), rank_correlation=round(e.rate.rank().corr(e.rate_broad.rank()), 2)))
print(pd.DataFrame(chk).to_string(index=False))

**Reading it:** in 2014–2021 the top 3 stays the same when the borderline statuses are counted (rank correlation about 0.97), so that ranking is stable. In 2022–2024 the top 3 changes (rank correlation about 0.6), so the 2022–24 ranking depends on how the borderline "Retired" cases are treated.

### 8.3 Answer
* **2014–2021 – highest:** Lotus F1 (17 %), Toro Rosso (16 %), Renault and McLaren (13 %). **Lowest:** Aston Martin (2 %), Alfa Romeo (3 %), Mercedes and Ferrari (4 %).
* **2022–2024 – highest:** Alfa Romeo (12.5 %) and Alpine (10 %). **Lowest:** Sauber (2 %), then McLaren, Red Bull and Aston Martin (3.7 %).
* **Overall,** mechanical failures became rarer: **9.0 % of starts → 5.5 %**. Mercedes was reliable in both eras (about 4 %); McLaren (13 % → 3.7 %) and Red Bull (11 % → 3.7 %) improved most; Alfa Romeo got worse (3 % → 12.5 %).
* **Caution:** the 2022–24 ranking is less stable than 2014–21 (§8.2). Teams have only 40–320 starts, so a few failures move a percentage a lot (e.g. Aston Martin's 2 % in 2014–21 is 1 failure in 44 starts). Teams that changed name (Force India → Racing Point → Aston Martin, Toro Rosso → AlphaTauri → RB, …) are separate rows, because the data gives each name its own `constructorId`.
* **Grain:** one row per race start (a driver in a race); each team has 2 per race. **Missingness:** every start has a status; "Retired" (reason not given) is borderline and tested above. Cars that broke down *before* the start are not race starts and are not counted. Also counting the 3 borderline statuses (Retired, Out of fuel, Debris) is tested in §8.2. By team decision, *Wheel nut* and *Seat* count as mechanical, while *Wheel, Puncture, Tyre, Undertray, wings and Damage* do not (they may be driver error or contact).
* **Pre-race cutoff:** status is only known **after** the race, so it can **never be a feature**. Q3 is analysis only; a team's reliability could later be used as a feature only if built from **earlier races**.

---
# Part 3 · Feature selection & split

## 9.1 Which years do we train on?
Qualifying data is the backbone of our starting-position features, so we checked how many drivers have it in each season.

In [ ]:
cov = (MODEL_BASE[KEY + ['year']].merge(CLEAN['qualifying'][KEY + ['position']].rename(columns={'position': 'q_pos'}), on=KEY, how='left', validate='one_to_one')
       .assign(has_q=lambda d: d.q_pos.notna()).groupby('year').has_q.mean() * 100).loc[1994:]
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(cov.index, cov.values, color=['#c9ccd1' if y < 2003 else (BLUE if y < 2006 else ORANGE) for y in cov.index], zorder=3)
ax.set_ylim(0, 112); ax.set_ylabel('% of drivers with a qualifying result'); ax.set_title('Qualifying coverage per season')
ax.text(1999, 66, '1996–2002\nincomplete\n(6–59 %)', ha='center', fontsize=8.5, color=INK2); ax.text(2004, 104, '2003–05\nsingle-lap', ha='center', fontsize=8, color=INK2); ax.text(2015, 104, '2006 onward: Q1/Q2/Q3 knockout', ha='center', fontsize=8.5, color=INK2)
save(fig, '15_qualifying_coverage_by_year')
print(f'1994–95: {cov.loc[:1995].min():.0f}–{cov.loc[:1995].max():.0f} %  |  1996–2002: {cov.loc[1996:2002].min():.0f}–{cov.loc[1996:2002].max():.0f} %  |  2003–2024: {cov.loc[2003:].min():.0f}–{cov.loc[2003:].max():.0f} %   (no qualifying data before 1994)')

**Decision: train on 2003–2019** (rows from 2003; earlier races still feed the form features).
- **Before 1994** there is no qualifying data at all; **1996–2002** it exists for only 6–59 % of drivers (1994–95 are nearly complete, then a long gap). The starting-position features would be mostly empty. The primer suggests restricting the window in this case.
- **From 2003** coverage is 99–100 %.
- **Knockout format only from 2006.** For 2003–05 the Q2/Q3 flags don't apply, so they are left empty and a flag `knockout_format` (0 / 1) tells the model which format applied.
- §10.2 re-trains from 1950, 1994, 2003 and 2006 to confirm the choice.

## 9.2 Features
**Rule:** a row may only use what is known **after qualifying and before the race**, plus results of **strictly earlier races**. §9.3 tests this.

| Group | Candidate features |
|---|---|
| A. Starting position | `grid_pos`, `quali_pos`, `gap_pct`, `reached_q2`, `reached_q3`, `knockout_format`, `field_size`, `pit_lane_start` |
| B. Driver form | podium rate, average finish, average points over the last N races; DNF rate; previous starts; age |
| C. Team form | team podiums and points over the last N races; team's average grid; team's mechanical-failure rate |
| D. Standings before the race | driver and team points, position, wins (after the **previous** round) |
| E. Circuit | circuit's past front-row conversion; driver's past podiums at this circuit |

In [ ]:
TRAIN_START, TRAIN_END, VAL_END = 2003, 2019, 2021          # train 2003–2019 | validation 2020–21 | test 2022–24
KNOCKOUT_FROM = 2006                                          # Q1/Q2/Q3 knockout qualifying exists from 2006
WINDOWS = (3, 5, 10)                                          # "last N races" windows we compare
PRIOR_RATE, PRIOR_K = 0.72, 10                                # circuit conversion is pulled towards 72 % (Q1) until the circuit has enough races

RACE_IDX  = {rid: i for i, rid in enumerate(CLEAN['races'].sort_values(['year', 'round']).raceId)}   # chronological order of all races
RACE_DATE = CLEAN['races'].set_index('raceId').date
RACE_YEAR = CLEAN['races'].set_index('raceId').year
BASE_COLS = ['raceId', 'driverId', 'constructorId', 'year', 'circuitId', 'grid', 'positionOrder', 'points', 'status', 'podium']

def prev_roll(s, n, how='mean'):
    """rolling statistic over the n races BEFORE the current one (shift(1) is what keeps the current race out)."""
    r = s.shift(1).rolling(n, min_periods=1)
    return r.mean() if how == 'mean' else r.sum()

def build_features(res, qual, dstand, cstand, drivers):
    d = res[BASE_COLS].copy()
    d['race_idx'] = d.raceId.map(RACE_IDX)

    # ── A. starting position (known after qualifying) ───────────────────────
    d['field_size'] = d.groupby('raceId').driverId.transform('size')
    d['pit_lane_start'] = (d.grid == 0).astype(int)
    d['grid_pos'] = np.where(d.grid > 0, d.grid, d.field_size + 1)       # pit-lane start = back of the field
    q = qual[['raceId', 'driverId', 'position', 'q1_s', 'q2_s', 'q3_s']].rename(columns={'position': 'quali_pos'}).copy()
    for c in ['q1_s', 'q2_s', 'q3_s']: q[c + '_best'] = q.groupby('raceId')[c].transform('min')       # fastest time of each session
    last_time = q.q3_s.fillna(q.q2_s).fillna(q.q1_s)                                                  # time in the last session he drove
    last_best = np.where(q.q3_s.notna(), q.q3_s_best, np.where(q.q2_s.notna(), q.q2_s_best, q.q1_s_best))
    q['gap_pct'] = (last_time / last_best - 1) * 100                                                  # % slower than the fastest in the same session
    q['reached_q2'] = q.q2_s.notna().astype(float); q['reached_q3'] = q.q3_s.notna().astype(float)
    d = d.merge(q[['raceId', 'driverId', 'quali_pos', 'gap_pct', 'reached_q2', 'reached_q3']], on=['raceId', 'driverId'], how='left', validate='one_to_one')
    d['knockout_format'] = (d.year >= KNOCKOUT_FROM).astype(int)                                        # 1 = Q1/Q2/Q3 knockout existed
    d.loc[d.knockout_format == 0, ['reached_q2', 'reached_q3']] = np.nan                                # not applicable before 2006 (not 'eliminated')

    # ── B. driver form (earlier races only) ─────────────────────────────────
    d['dnf'] = (~(d.status.eq('Finished') | d.status.str.startswith('+'))).astype(int)
    d['mech'] = d.status.isin(MECHANICAL).astype(int)
    d = d.sort_values(['driverId', 'race_idx']); g = d.groupby('driverId')
    for n in WINDOWS:
        d[f'drv_podium_{n}'] = g.podium.transform(lambda s: prev_roll(s, n))
        d[f'drv_finish_{n}'] = g.positionOrder.transform(lambda s: prev_roll(s, n))
        d[f'drv_points_{n}'] = g.points.transform(lambda s: prev_roll(s, n))
    d['drv_dnf_5'] = g.dnf.transform(lambda s: prev_roll(s, 5))
    d['drv_starts'] = g.cumcount()
    d['drv_age'] = (d.raceId.map(RACE_DATE) - d.driverId.map(drivers.set_index('driverId').dob)).dt.days / 365.25

    # ── C. constructor form (one row per team per race, then earlier races only) ──
    cr = (d.groupby(['raceId', 'constructorId'])
           .agg(race_idx=('race_idx', 'first'), c_pod=('podium', 'sum'), c_pts=('points', 'sum'), c_grid=('grid_pos', 'mean'), c_mech=('mech', 'sum'), c_n=('mech', 'size'))
           .reset_index().sort_values(['constructorId', 'race_idx']))
    gc = cr.groupby('constructorId')
    for n in WINDOWS:
        cr[f'con_podium_{n}'] = gc.c_pod.transform(lambda s: prev_roll(s, n))
        cr[f'con_points_{n}'] = gc.c_pts.transform(lambda s: prev_roll(s, n))
    cr['con_grid_5'] = gc.c_grid.transform(lambda s: prev_roll(s, 5))
    cr['con_mech_10'] = gc.c_mech.transform(lambda s: prev_roll(s, 10, 'sum')) / gc.c_n.transform(lambda s: prev_roll(s, 10, 'sum'))
    keep_c = ['raceId', 'constructorId'] + [c for c in cr.columns if c.startswith('con_')]
    d = d.merge(cr[keep_c], on=['raceId', 'constructorId'], how='left', validate='many_to_one')

    # ── D. standings before the race = standings after the PREVIOUS round of the same season ──
    for tab, key, pre in [(dstand, 'driverId', 'drv_st_'), (cstand, 'constructorId', 'con_st_')]:
        s = tab[['raceId', key, 'points', 'position', 'wins']].copy()
        s['race_idx'] = s.raceId.map(RACE_IDX); s['year'] = s.raceId.map(RACE_YEAR); s = s.sort_values([key, 'race_idx'])
        for c in ['points', 'position', 'wins']: s[pre + c] = s.groupby([key, 'year'])[c].shift(1)
        d = d.merge(s[['raceId', key, pre + 'points', pre + 'position', pre + 'wins']], on=['raceId', key], how='left', validate='one_to_one' if key == 'driverId' else 'many_to_one')
        for c in ['points', 'wins']: d[pre + c] = d[pre + c].fillna(0)            # nothing scored yet this season
        # (position stays empty before a season's first race)

    # ── E. circuit history (earlier races only) ─────────────────────────────
    fr = d.grid_pos <= 2
    cc = (d.assign(fr_n=fr.astype(int), fr_p=(fr & (d.podium == 1)).astype(int)).groupby(['raceId', 'circuitId'])
           .agg(race_idx=('race_idx', 'first'), fr_n=('fr_n', 'sum'), fr_p=('fr_p', 'sum')).reset_index().sort_values(['circuitId', 'race_idx']))
    gcir = cc.groupby('circuitId')
    cum_n = gcir.fr_n.cumsum() - cc.fr_n; cum_p = gcir.fr_p.cumsum() - cc.fr_p                 # totals BEFORE this race
    cc['circ_conv'] = (cum_p + PRIOR_K * PRIOR_RATE) / (cum_n + PRIOR_K)
    d = d.merge(cc[['raceId', 'circuitId', 'circ_conv']], on=['raceId', 'circuitId'], how='left', validate='many_to_one')
    d = d.sort_values(['driverId', 'circuitId', 'race_idx'])
    d['drv_circ_podiums'] = d.groupby(['driverId', 'circuitId']).podium.cumsum() - d.podium

    d['split'] = np.select([d.year <= TRAIN_END, d.year <= VAL_END], ['train', 'val'], 'test')
    return d.sort_values(['race_idx', 'driverId']).reset_index(drop=True)

assert MODEL_BASE.year.max() == 2024, 'dataset should end in 2024'
ALL = build_features(MODEL_BASE, CLEAN['qualifying'], CLEAN['driver_standings'], CLEAN['constructor_standings'], CLEAN['drivers'])
FEAT = ALL[ALL.year >= TRAIN_START].copy()            # history comes from ALL earlier seasons; the modelling rows start in 2003
assert not FEAT.duplicated(KEY).any()
print(f'feature table: {len(FEAT):,} rows × {FEAT.shape[1]} columns   (history before {TRAIN_START} is used for form, but rows start in {TRAIN_START})')
print(FEAT.groupby('split').agg(rows=('podium', 'size'), races=('raceId', 'nunique'), seasons=('year', lambda s: f'{s.min()}–{s.max()}'), podium_rate=('podium', 'mean')).round(3).to_string())

### 9.3 Leakage test
If a feature is leak-free, **scrambling the current race and every later race must not change it**. We scramble them (and the standings tables), rebuild all features, and compare. A deliberately leaky feature (no `shift(1)`) proves the test can catch a leak.

In [ ]:
FEATURE_COLS = [c for c in ALL.columns if c.startswith(('drv_', 'con_', 'circ_')) or c in
                ['grid_pos', 'quali_pos', 'gap_pct', 'reached_q2', 'reached_q3', 'knockout_format', 'field_size', 'pit_lane_start']]

def scramble_from(idx_from, seed=0):
    rng = np.random.default_rng(seed)
    r = MODEL_BASE.copy(); m = (r.raceId.map(RACE_IDX) >= idx_from).values
    perm = rng.permutation(m.sum())
    for c in ['positionOrder', 'points', 'status', 'podium']: r.loc[m, c] = r.loc[m, c].values[perm]      # same shuffle for all outcome columns
    out = {}
    for name in ['driver_standings', 'constructor_standings']:
        t = CLEAN[name].copy(); mt = (t.raceId.map(RACE_IDX) >= idx_from).values
        for c in ['points', 'position', 'wins']: t.loc[mt, c] = rng.permutation(t.loc[mt, c].values)
        out[name] = t
    return r, out['driver_standings'], out['constructor_standings']

def leaky_demo(res):                       # NOT used for modelling: same as drv_podium_5 but WITHOUT shift(1)
    d = res[['raceId', 'driverId', 'podium']].copy(); d['race_idx'] = d.raceId.map(RACE_IDX); d = d.sort_values(['driverId', 'race_idx'])
    d['leaky'] = d.groupby('driverId').podium.transform(lambda s: s.rolling(5, min_periods=1).mean()); return d.set_index(['raceId', 'driverId']).leaky

base_leaky = leaky_demo(MODEL_BASE); rows = []
for year, rnd in [(2012, 8), (2019, 12), (2023, 5)]:
    rc = CLEAN['races']; rid = rc.loc[(rc['year'] == year) & (rc['round'] == rnd), 'raceId'].iloc[0]; idx = RACE_IDX[rid]
    r, ds, cs = scramble_from(idx)
    P = build_features(r, CLEAN['qualifying'], ds, cs, CLEAN['drivers'])
    upto = ALL.race_idx <= idx                                             # this race and everything before it
    a = ALL[upto].set_index(KEY)[FEATURE_COLS]; b = P[P.race_idx <= idx].set_index(KEY)[FEATURE_COLS].loc[a.index]
    changed = [c for c in FEATURE_COLS if not np.allclose(a[c].astype(float), b[c].astype(float), equal_nan=True)]
    lk = leaky_demo(r); m_ = lk.index.get_level_values('raceId').map(RACE_IDX) == idx
    leak_changed = not np.allclose(base_leaky[m_], lk[m_], equal_nan=True)
    rows.append(dict(race=f'{year} round {rnd}', feature_rows_checked=len(a), real_features_that_changed=len(changed), leaky_demo_detected=leak_changed))
    assert not changed, changed
res_leak = pd.DataFrame(rows); print(res_leak.to_string(index=False))
print(f'\n✔ none of the {len(FEATURE_COLS)} features changed when the current and all later results were scrambled; the deliberately leaky feature was caught every time')

### 9.4 Missing values (training years)
Filled later inside the model pipeline, using training years only. Empty `reached_q2/q3` = 2003–05 (not applicable); empty form = a driver's or team's first races.

In [ ]:
TR = FEAT[FEAT.split == 'train']
miss = TR[FEATURE_COLS].isna().mean().sort_values(ascending=False); miss = miss[miss > 0]
print((miss * 100).round(1).rename('% empty').to_string())

### 9.5 Which "last N races" window? (training years)
3, 5 and 10 races give almost the same correlation, so the choice is not critical. The final list fixes one window per feature (a short window for podium counts, a longer one for average finish); the table shows the exact choice hardly matters.

In [ ]:
fam = {'drv_podium': 'driver podium rate', 'drv_finish': 'driver average finish', 'drv_points': 'driver average points', 'con_podium': 'team podiums', 'con_points': 'team points'}
wt = pd.DataFrame({f: {n: TR[f'{f}_{n}'].corr(TR.podium, method='spearman') for n in WINDOWS} for f in fam}).T
wt.index = [fam[i] for i in wt.index]; wt.columns = [f'last {n}' for n in WINDOWS]
print('Spearman correlation with "finished on the podium" (training years)'); print(wt.round(3).to_string())
BEST_N = {f: max(WINDOWS, key=lambda n: abs(TR[f'{f}_{n}'].corr(TR.podium, method='spearman'))) for f in fam}
print('\nwindow with the strongest correlation:', {fam[f]: n for f, n in BEST_N.items()})

### 9.6 Correlation maps (training years)
Bars: how strongly each feature moves with the podium. Matrix: which features repeat each other.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
GROUPS = {'A': ['grid_pos', 'quali_pos', 'gap_pct', 'reached_q2', 'reached_q3', 'knockout_format', 'field_size', 'pit_lane_start'],
          'B': ['drv_starts', 'drv_age', 'drv_dnf_5', 'drv_podium_5', 'drv_finish_10', 'drv_points_5'],
          'C': ['con_grid_5', 'con_mech_10', 'con_podium_3', 'con_points_5'],
          'D': ['drv_st_points', 'drv_st_position', 'drv_st_wins', 'con_st_points', 'con_st_position', 'con_st_wins'],
          'E': ['circ_conv', 'drv_circ_podiums']}
FEATURES = [f for g_ in GROUPS.values() for f in g_]
GROUP_OF = {f: g_ for g_, fs in GROUPS.items() for f in fs}
GROUP_COLOR = {'A': BLUE, 'B': ORANGE, 'C': AQUA, 'D': MAGENTA, 'E': INK2}

cor_t = TR[FEATURES].apply(lambda s: s.corr(TR.podium, method='spearman')).sort_values(key=abs)
fig, ax = plt.subplots(figsize=(7.5, 8.2))
ax.barh(range(len(cor_t)), cor_t.values, color=[GROUP_COLOR[GROUP_OF[f]] for f in cor_t.index], height=0.7, zorder=3)
for i, v in enumerate(cor_t.values): ax.text(v + (0.012 if v >= 0 else -0.012), i, f'{v:+.2f}', va='center', ha='left' if v >= 0 else 'right', fontsize=8, zorder=5)
ax.set_yticks(range(len(cor_t))); ax.set_yticklabels(cor_t.index, fontsize=8.5); ax.axvline(0, color=INK2, lw=0.8); ax.set_xlim(-0.62, 0.62)
ax.set_xlabel('Spearman correlation with "finished on the podium"  (training years)'); ax.set_title('Which features move with the podium?')
from matplotlib.patches import Patch
LAB = {'A': 'A start position', 'B': 'B driver form', 'C': 'C team form', 'D': 'D standings', 'E': 'E circuit'}
ax.legend(handles=[Patch(color=GROUP_COLOR[k], label=LAB[k]) for k in LAB], loc='lower right', fontsize=8); save(fig, '16_feature_correlation_with_podium')

cm = TR[FEATURES].corr(method='spearman'); cmap = LinearSegmentedColormap.from_list('bw', [BLUE, '#ffffff', ORANGE])
fig, ax = plt.subplots(figsize=(12.5, 10.5))
im = ax.imshow(cm.values, cmap=cmap, vmin=-1, vmax=1)
ax.set_xticks(range(len(cm))); ax.set_xticklabels(cm.columns, rotation=60, ha='right', fontsize=8); ax.set_yticks(range(len(cm))); ax.set_yticklabels(cm.index, fontsize=8)
for i in range(len(cm)):
    for j in range(len(cm)): ax.text(j, i, f'{cm.values[i, j]:.1f}', ha='center', va='center', fontsize=6, color='white' if abs(cm.values[i, j]) > 0.75 else INK)
ax.grid(False); plt.colorbar(im, ax=ax, shrink=0.7, label='Spearman correlation'); ax.set_title('Do the features say the same thing?  (blue = opposite, orange = same direction)')
save(fig, '17_feature_correlation_matrix')

pairs = cm.where(np.triu(np.ones(cm.shape, bool), 1)).stack(); strong = pairs[pairs.abs() >= 0.8].sort_values(key=abs, ascending=False)
print(f'{len(strong)} feature pairs with |correlation| ≥ 0.8:'); print(strong.round(2).to_string())

## 9.7 Feature importance and final list
Correlation with the podium (training years) gives each feature's strength. **Strong** ≥ 0.40 · **medium** 0.20–0.40 · **weak** 0.05–0.20 · **none** < 0.05.

In [ ]:
FINAL_FEATURES = ['grid_pos', 'quali_pos', 'gap_pct', 'reached_q2', 'reached_q3', 'knockout_format',                      # A  start position
                  'drv_podium_5', 'drv_finish_10', 'drv_points_5', 'drv_dnf_5', 'drv_starts',                                # B  driver form
                  'con_podium_3', 'con_points_5', 'con_grid_5', 'con_mech_10',                                              # C  team form
                  'drv_st_points', 'drv_st_position', 'drv_st_wins', 'con_st_points', 'con_st_position', 'con_st_wins',    # D  standings before the race
                  'drv_circ_podiums']                                                                                        # E  circuit
DROPPED = {'drv_age': 'repeats drv_starts (0.81)', 'field_size': 'no signal', 'pit_lane_start': 'no signal, very rare', 'circ_conv': 'no signal'}
assert set(FINAL_FEATURES) <= set(FEAT.columns) and len(FINAL_FEATURES) == len(set(FINAL_FEATURES))
MEANING = {'grid_pos': 'starting slot (pit lane = back)', 'quali_pos': 'qualifying position', 'gap_pct': '% slower than fastest, same session',
 'reached_q2': 'got a Q2 time', 'reached_q3': 'got a Q3 time', 'knockout_format': 'Q1/Q2/Q3 format existed (2006+)', 'field_size': 'cars in the race', 'pit_lane_start': 'started from pit lane',
 'drv_podium_5': 'driver podium rate, last 5', 'drv_finish_10': 'driver avg finish, last 10', 'drv_points_5': 'driver avg points, last 5', 'drv_dnf_5': 'driver DNF rate, last 5',
 'drv_starts': 'driver previous starts', 'drv_age': 'driver age',
 'con_podium_3': 'team podiums, last 3 races', 'con_points_5': 'team points, last 5 races', 'con_grid_5': 'team avg grid, last 5', 'con_mech_10': 'team mechanical-failure rate, last 10',
 'drv_st_points': 'driver season points', 'drv_st_position': 'driver championship position', 'drv_st_wins': 'driver season wins',
 'con_st_points': 'team season points', 'con_st_position': 'team championship position', 'con_st_wins': 'team season wins',
 'drv_circ_podiums': "driver's earlier podiums here", 'circ_conv': "circuit's past front-row conversion"}
imp = pd.DataFrame({'feature': list(MEANING)})
imp['group'] = imp.feature.map(GROUP_OF); imp['meaning'] = imp.feature.map(MEANING)
imp['corr'] = imp.feature.map(lambda f: TR[f].corr(TR.podium, method='spearman'))
imp['strength'] = pd.cut(imp['corr'].abs(), [-1, .05, .20, .40, 1], labels=['none', 'weak', 'medium', 'strong'])
imp['decision'] = imp.feature.map(lambda f: 'drop: ' + DROPPED[f] if f in DROPPED else ('keep: format flag, not a predictor' if f == 'knockout_format' else 'keep'))
imp = imp.sort_values(['group', 'corr'], key=lambda c: c if c.name == 'group' else -c.abs()).reset_index(drop=True)
print(imp.assign(corr=imp['corr'].round(2)).to_string(index=False))

**Reading the table**
- **Strong (≈ 0.5):** starting position, the driver's and team's recent podiums, team wins this season — they say how fast the car and driver are.
- **Medium:** reached Q3/Q2, season points, previous podiums at the circuit, previous starts.
- **Weak, kept:** team's mechanical-failure rate, driver's DNF rate. They are the only features about *reliability*; the ablation will confirm whether they help.
- **`quali_pos` is kept next to `grid_pos`:** they differ for some drivers (penalties, knock-on moves) and the check below compares the pair with each alone.
- **Overlap:** standings and form say similar things (|corr| ≈ 0.8–0.9). Fine for trees and the neural network; the ablation will show if logistic regression suffers.
- **Dropped:** `drv_age`, `field_size`, `pit_lane_start`, `circ_conv`.
- **`reached_q2` / `reached_q3`** overlap with `quali_pos` (a driver knocked out in Q1 qualifies low), so their sign in the explanations (§13) can look odd.

In [ ]:
# checks behind the statements above (training years)
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
VA_ = FEAT[FEAT.split == 'val']
diff_share = (TR.grid_pos != TR.quali_pos)[TR.quali_pos.notna()].mean()
print(f'drivers whose starting slot differs from their qualifying position: {diff_share:.0%}')
print(f'pit-lane starters among training rows: {TR.pit_lane_start.mean():.1%}')
def lr_score(cols):
    m = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=1000)).fit(TR[cols], TR.podium)
    return average_precision_score(VA_.podium, m.predict_proba(VA_[cols])[:, 1])
print('logistic regression, validation PR-AUC:', {' + '.join(c): round(lr_score(c), 3) for c in (['grid_pos'], ['quali_pos'], ['grid_pos', 'quali_pos'])})

Differences of a few hundredths are within validation noise, so `quali_pos` is kept as a second description of the start (it is not a leak: both are known before the race), not because the gain is large.

## 9.8 Train / validation / test split

In [ ]:
MODEL_TABLE = FEAT[KEY + ['year', 'split'] + FINAL_FEATURES + ['podium']].reset_index(drop=True)
sp = MODEL_TABLE.groupby('split').agg(seasons=('year', lambda s: f'{s.min()}–{s.max()}'), races=('raceId', 'nunique'), rows=('podium', 'size'), podium_rate=('podium', 'mean')).loc[['train', 'val', 'test']]
sp['share_of_rows'] = (sp.rows / sp.rows.sum()).round(2); sp['podium_rate'] = sp.podium_rate.round(3)
print(f'{len(FINAL_FEATURES)} features | modelling table {MODEL_TABLE.shape[0]:,} rows x {MODEL_TABLE.shape[1]} columns'); print(sp.to_string())

**Chronological split:** train 2003–2019 · validation 2020–2021 · test 2022–2024 (the boundaries suggested in the primer).
- **Why not random:** our features use earlier races, so a random split would put a 2018 race in training and a 2017 race in test — the model would learn from the future, and drivers of the same race would sit on both sides. Scores would look better than they are.
- **Why chronological:** it copies real use — predict an upcoming race from the past.
- **Roles:** *train* fits the models and the imputers/scalers · *validation* picks the threshold and settings · *test* is used once, at the end.
- **Caveats:** 2020 was a shortened season; 2022 brought new technical rules, so the test is deliberately a harder, "new era" check. The podium rate is stable (14–15 %) across all three sets.

---
# Part 4 · Training

## 10.1 Setup
Same rows and same 22 features for every model. Median filling, a "was missing" flag and scaling are **fitted on the training rows only**. Scores below are on the **validation years (2020–21)**; the test years stay untouched.

| Model | Role |
|---|---|
| Baseline | logistic regression on `grid_pos` alone — the bar to beat |
| Logistic regression | statistical model 1 |
| Gradient boosting | statistical model 2 |
| Neural network (PyTorch) | one hidden layer, dropout, early stopping; averaged over 5 seeds (size tuned in §10.3) |

Metrics here are **PR-AUC** (main — podiums are a small share of rows) and **ROC-AUC**. Both score the ranking, so no threshold is needed yet.

In [ ]:
import torch, torch.nn as nn
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
torch.set_num_threads(2)

from sklearn.base import BaseEstimator, ClassifierMixin
class TorchMLP(ClassifierMixin, BaseEstimator):
    """Shallow network: inputs -> 32 ReLU units (dropout) -> 1. sklearn-style fit / predict_proba. Early stopping on the NEWEST 15 % of training rows."""
    def __init__(self, seed=0, hidden=32, dropout=0.2, lr=1e-3, wd=1e-3, max_epochs=80, patience=8):
        self.seed, self.hidden, self.dropout, self.lr, self.wd, self.max_epochs, self.patience = seed, hidden, dropout, lr, wd, max_epochs, patience
        self.classes_ = np.array([0, 1])
    def fit(self, X, y):
        torch.manual_seed(self.seed); X = np.asarray(X, 'float32'); y = np.asarray(y, 'float32'); n_es = int(len(X) * 0.15)
        Xtr, ytr, Xes, yes = (torch.tensor(a) for a in (X[:-n_es], y[:-n_es], X[-n_es:], y[-n_es:]))   # rows arrive in time order
        self.net = nn.Sequential(nn.Linear(X.shape[1], self.hidden), nn.ReLU(), nn.Dropout(self.dropout), nn.Linear(self.hidden, 1))
        opt = torch.optim.Adam(self.net.parameters(), lr=self.lr, weight_decay=self.wd); lossf = nn.BCEWithLogitsLoss(); best, bad, state = 1e9, 0, None
        for _ in range(self.max_epochs):
            self.net.train(); perm = torch.randperm(len(Xtr))
            for i in range(0, len(Xtr), 256):
                b = perm[i:i + 256]; opt.zero_grad(); lossf(self.net(Xtr[b]).squeeze(1), ytr[b]).backward(); opt.step()
            self.net.eval()
            with torch.no_grad(): v = lossf(self.net(Xes).squeeze(1), yes).item()
            if v < best - 1e-4: best, bad, state = v, 0, {k: t.clone() for k, t in self.net.state_dict().items()}
            else:
                bad += 1
                if bad >= self.patience: break
        self.net.load_state_dict(state); self.net.eval(); self.fitted_ = True; return self
    def predict_proba(self, X):
        with torch.no_grad(): p = torch.sigmoid(self.net(torch.tensor(np.asarray(X, 'float32'))).squeeze(1)).numpy()
        return np.c_[1 - p, p]

prep = lambda: make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler())
def make_model(name, seed=0, params=None):
    p = params or {}
    if name == 'Baseline (grid only)': return make_pipeline(ColumnTransformer([('g', 'passthrough', ['grid_pos'])]), prep(), LogisticRegression())
    if name == 'Logistic regression':  return make_pipeline(prep(), LogisticRegression(C=p.get('C', 1.0), max_iter=3000))
    if name == 'Gradient boosting':    return HistGradientBoostingClassifier(max_depth=p.get('max_depth', 3), learning_rate=p.get('learning_rate', 0.05), max_iter=p.get('max_iter', 150), random_state=seed, early_stopping=False)
    if name == 'Neural network':       return make_pipeline(prep(), TorchMLP(seed=seed, hidden=p.get('hidden', 32), dropout=p.get('dropout', 0.2)))
MODELS = ['Baseline (grid only)', 'Logistic regression', 'Gradient boosting', 'Neural network']
VAL = ALL[ALL.split == 'val']

def train_rows(start): return ALL[(ALL.year >= start) & (ALL.split == 'train')].sort_values('race_idx')
def score(name, start, seed=0):
    tr = train_rows(start); m = make_model(name, seed).fit(tr[FINAL_FEATURES], tr.podium); p = m.predict_proba(VAL[FINAL_FEATURES])[:, 1]
    return average_precision_score(VAL.podium, p), roc_auc_score(VAL.podium, p), p

## 10.2 Which start year? 1950 vs 1994 vs 2003 vs 2006
Same models (default settings), same validation years — only the first training season changes. 1950 = all data; 1994 = first season with qualifying data (patchy 1996–2002).

In [ ]:
STARTS = [1950, 1994, 2003, 2006]; SEEDS = range(5); rows = []; preds = {}
for st in STARTS:
    for name in MODELS:
        runs = [score(name, st, s) for s in (SEEDS if name == 'Neural network' else [0])]
        pr, roc = [r[0] for r in runs], [r[1] for r in runs]; preds[(st, name)] = runs[0][2]
        rows.append(dict(start=st, train_rows=len(train_rows(st)), model=name, PR_AUC=np.mean(pr), PR_sd=np.std(pr), ROC_AUC=np.mean(roc)))
res = pd.DataFrame(rows)
print('PR-AUC on validation 2020–21 (neural network = mean of 5 seeds)'); print(res.pivot(index='start', columns='model', values='PR_AUC')[MODELS].round(3).to_string())
print('\nROC-AUC'); print(res.pivot(index='start', columns='model', values='ROC_AUC')[MODELS].round(3).to_string())
print('\ntraining rows:', res.drop_duplicates('start').set_index('start').train_rows.to_dict())

# how much does a score move just by luck? resample the validation RACES 300 times (logistic regression, 2003 start)
rng = np.random.default_rng(0); races = VAL.raceId.unique(); p0 = pd.Series(preds[(2003, 'Logistic regression')], index=VAL.index); bs = []
for _ in range(300):
    pick = rng.choice(races, len(races)); idx = np.concatenate([VAL.index[VAL.raceId == r] for r in pick]); bs.append(average_precision_score(VAL.loc[idx, 'podium'], p0.loc[idx]))
NOISE = float(np.std(bs)); print(f'\nvalidation noise (bootstrap over races): ±{NOISE:.3f} PR-AUC')

fig, ax = plt.subplots(figsize=(8.5, 3.8)); w = 0.2
for i, st in enumerate(STARTS):
    d = res[res.start == st].set_index('model').loc[MODELS]
    ax.bar(np.arange(len(MODELS)) + (i - 1.5) * w, d.PR_AUC, w, yerr=d.PR_sd, color=['#8a8f98', '#c9ccd1', BLUE, ORANGE][i], label=f'from {st}', zorder=3, capsize=2)
ax.set_xticks(range(len(MODELS))); ax.set_xticklabels(MODELS, fontsize=9); ax.set_ylim(0.5, 0.8); ax.set_ylabel('PR-AUC (validation)'); ax.set_title('Does the training start year matter?')
ax.legend(fontsize=8, loc='upper left', ncol=4); save(fig, '18_training_start_year')

**Result**
- **Every model beats the grid-only baseline** in the table above.
- **The start year barely matters.** Changes from one start year to the next are smaller than the validation noise (bootstrap noise printed with the table). No start year is clearly better.
- **Why not 1950 or 1994 then?** They add rows but no gain, and qualifying is empty for most of those rows (none before 1994, 6–59 % in 1996–2002), so the starting-position features would be mostly filled-in guesses. 1950s racing (cars, reliability, 24-car fields) is also very different from today.
- The baseline is identical by construction — it only ranks drivers by grid slot.

**Decision: keep 2003.** Since scores don't separate the options, we choose on data quality: 2003 is the first season from which qualifying data is complete and continuous, and it keeps more rows than a 2006 start.

## 10.3 Tuning the models
So far every model used settings we picked by hand. Here we try a few settings per model and keep the best on the **validation years** (PR-AUC). If two settings are within 0.01 of each other, we take the **simpler** one, because on the small validation set a 0.01 gap is luck.

| Model | What we vary |
|---|---|
| Logistic regression | regularisation strength `C` (smaller = more restrained) |
| Gradient boosting | tree depth, learning rate (with enough trees for each rate) |
| Neural network | hidden units, dropout (3 seeds per setting) |

In [ ]:
TR03 = train_rows(2003)
GRIDS = {'Logistic regression': [dict(C=c) for c in (0.01, 0.1, 1, 10)],
         'Gradient boosting': [dict(max_depth=d, learning_rate=lr, max_iter=it) for d in (2, 3, 4) for lr, it in ((0.03, 300), (0.1, 100))],
         'Neural network': [dict(hidden=h, dropout=dr) for h in (8, 16, 32, 64) for dr in (0.1, 0.3)]}
DEFAULTS = {'Logistic regression': dict(C=1.0), 'Gradient boosting': dict(max_depth=3, learning_rate=0.05, max_iter=150), 'Neural network': dict(hidden=32, dropout=0.2)}
COMPLEXITY = {'Logistic regression': lambda q: q['C'], 'Gradient boosting': lambda q: (q['max_depth'], q['max_iter'] * q['learning_rate']), 'Neural network': lambda q: (q['hidden'], -q['dropout'])}

def tune_score(name, params, seeds):
    ps, rs = [], []
    for sd in seeds:
        m = make_model(name, sd, params).fit(TR03[FINAL_FEATURES], TR03.podium); p = m.predict_proba(VAL[FINAL_FEATURES])[:, 1]
        ps.append(average_precision_score(VAL.podium, p)); rs.append(roc_auc_score(VAL.podium, p))
    return np.mean(ps), np.mean(rs)

TUNED, tune_tables = {}, {}
for name in GRIDS:
    cands = GRIDS[name] + ([DEFAULTS[name]] if DEFAULTS[name] not in GRIDS[name] else [])
    seeds = range(3) if name == 'Neural network' else [0]
    t = pd.DataFrame([dict(setting=', '.join(f'{k}={v}' for k, v in q.items()), default=(q == DEFAULTS[name]), PR_AUC=(sc := tune_score(name, q, seeds))[0], ROC_AUC=sc[1], _q=q) for q in cands])
    best = t.PR_AUC.max(); ok = t[t.PR_AUC >= best - 0.01]
    TUNED[name] = min(ok._q, key=COMPLEXITY[name]); t['chosen'] = [q == TUNED[name] for q in t._q]; tune_tables[name] = t.drop(columns='_q')
    print(f'--- {name}  (best PR-AUC {best:.3f}; chosen = simplest within 0.01 of it)'); print(tune_tables[name].round(3).to_string(index=False)); print()
pd.concat(tune_tables, names=['model']).reset_index(level=0).to_csv(PROC_DIR / 'tuning_results.csv', index=False)
print('chosen settings:', TUNED)

In [ ]:
FITTED = {n: [make_model(n, s, TUNED.get(n)).fit(TR03[FINAL_FEATURES], TR03.podium) for s in (SEEDS if n == 'Neural network' else [0])] for n in MODELS}
proba = lambda n, df: np.mean([m.predict_proba(df[FINAL_FEATURES])[:, 1] for m in FITTED[n]], axis=0)
P_TR = {n: proba(n, TR03) for n in MODELS}; P_VA = {n: proba(n, VAL) for n in MODELS}
d0 = res[res.start == 2003].set_index('model')
cmp_ = pd.DataFrame({'default PR-AUC': d0.PR_AUC, 'tuned PR-AUC': {n: average_precision_score(VAL.podium, P_VA[n]) for n in MODELS}, 'default ROC-AUC': d0.ROC_AUC, 'tuned ROC-AUC': {n: roc_auc_score(VAL.podium, P_VA[n]) for n in MODELS}}).loc[MODELS]
cmp_['PR-AUC change'] = cmp_['tuned PR-AUC'] - cmp_['default PR-AUC']
print('Validation scores, default vs tuned settings'); print(cmp_.round(3).to_string())

**What we changed**

| Model | Default → chosen setting |
|---|---|
| Logistic regression | `C` 1 → **0.1** (more restrained) |
| Gradient boosting | depth 3, rate 0.05, 150 trees → depth **2**, rate **0.03**, **300** trees (no early stopping, so every start year is trained the same way) |
| Neural network | 32 units, dropout 0.2 → **8 units, dropout 0.3** |

- The tuning table above gives the scores. Tuning moves them only slightly compared with the validation noise, so the models are not sensitive to these settings, and we keep the simpler settings because they do as well.
- Shallower gradient-boosting trees overfit less (the train/validation gap shrinks).
- A larger neural network can look slightly better on paper, but the smaller one is within 0.01, so we take it.

## 10.4 Choosing the threshold (again, for the tuned models)
A model's probability becomes "podium: yes" when it is **at or above a threshold**. A low threshold names many drivers (more podiums caught, more wrong names); a high one names few. We try thresholds from 0.05 to 0.90 on the **validation years** and keep, for each model, the one with the best **F1**.

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score
THR = np.round(np.arange(0.05, 0.91, 0.01), 2)
def at(y, p, t): pred = p >= t; return dict(precision=precision_score(y, pred, zero_division=0), recall=recall_score(y, pred), F1=f1_score(y, pred), podiums_named_per_race=pred.sum() / VAL.raceId.nunique())
sweep = {n: pd.DataFrame([dict(threshold=t, **at(VAL.podium, P_VA[n], t)) for t in THR]).set_index('threshold') for n in MODELS}
BEST_T = {n: float(sweep[n].F1.idxmax()) for n in MODELS}
EXPLAIN_MODEL = 'Logistic regression'   # declared in advance: the simplest model that is within noise of the best (see 10.3)

show = pd.DataFrame({n: sweep[n].F1 for n in MODELS}).loc[[0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]]
print('F1 on validation at different thresholds'); print(show.round(3).to_string())
print('\nbest threshold per model (max F1):', {n: f'{BEST_T[n]:.2f} (F1 {sweep[n].F1.max():.3f})' for n in MODELS})
print(f'\n{EXPLAIN_MODEL}: what each threshold does')
print(sweep[EXPLAIN_MODEL].loc[[0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]].round(3).to_string())

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for n, c in zip(MODELS, ['#8a8f98', BLUE, ORANGE, MAGENTA]):
    ax[0].plot(sweep[n].index, sweep[n].F1, color=c, label=n, lw=1.8); ax[0].plot(BEST_T[n], sweep[n].F1.max(), 'o', color=c)
ax[0].set_xlabel('threshold'); ax[0].set_ylabel('F1 (validation)'); ax[0].set_title('F1 by threshold (dot = best)'); ax[0].legend(fontsize=7.5)
d = sweep[EXPLAIN_MODEL]; ax[1].plot(d.index, d.precision, color=BLUE, label='precision'); ax[1].plot(d.index, d.recall, color=ORANGE, label='recall'); ax[1].plot(d.index, d.F1, color=INK2, label='F1', ls='--')
ax[1].axvline(BEST_T[EXPLAIN_MODEL], color=INK2, lw=0.8, ls=':'); ax[1].set_xlabel('threshold'); ax[1].set_title(f'{EXPLAIN_MODEL}: precision vs recall'); ax[1].legend(fontsize=8)
save(fig, '19_threshold_sweep')

**What the threshold does** (neural network table above)
- A **low** threshold names many drivers per race: most podium finishers are caught, but many names are wrong.
- A **high** threshold names few: almost every name is right, but most podium finishers are missed.
- The best thresholds are **well below 0.5**: podiums are rare, so probabilities stay modest. F1 is fairly flat around its maximum, so the exact value is not critical. Each model keeps its best-F1 threshold, **chosen on validation only**.

## 10.5 Full metrics (tuned models, best threshold per model)

In [ ]:
rows = []
for n in MODELS:
    for lab, df, P in [('train', TR03, P_TR), ('validation', VAL, P_VA)]:
        y, p = df.podium, P[n]; pred = p >= BEST_T[n]
        rows.append(dict(model=n, set=lab, threshold=BEST_T[n], ROC_AUC=roc_auc_score(y, p), PR_AUC=average_precision_score(y, p), F1=f1_score(y, pred), precision=precision_score(y, pred), recall=recall_score(y, pred)))
met = pd.DataFrame(rows); print(met.round(3).to_string(index=False))
met.to_csv(PROC_DIR / 'metrics_train_validation.csv', index=False)

**Reading the table**
- The three real models are close to each other on every metric.
- **The grid-only baseline is hard to beat:** the F1 gap to the models is small, and the extra features help PR-AUC somewhat more than F1, because the starting grid alone already finds most podium finishers.
- No strong sign of overfitting: training and validation scores are similar (gradient boosting usually has the largest gap).
- Validation numbers use a threshold tuned on validation, so they are slightly optimistic. The **test years** will give the honest figure.

## 10.6 What goes in, what comes out
- **In:** one row per (race, driver) with the final features (§9.7). Saved as `training_table.csv` (2003–2019, with the target `podium`).
- **Explanation model:** all four models are reported equally. For the worked examples in §13 we use **logistic regression**, declared in advance by the simplicity rule: when models are within noise of each other, take the simplest. This choice does not depend on test scores.
- **Out:** a **probability between 0 and 1** for every driver in every race ("how likely is a podium"), from each tuned model. The yes/no answer is "probability ≥ the model's threshold" (§10.4). Saved as `validation_predictions.csv`.

In [ ]:
nm = CLEAN['drivers'].assign(driver=lambda d: d.forename + ' ' + d.surname).set_index('driverId').driver
rc = CLEAN['races'].set_index('raceId')[['year', 'round', 'name']].rename(columns={'name': 'race'})
label = lambda d: d.assign(driver=d.driverId.map(nm), race=d.raceId.map(rc.race), round=d.raceId.map(rc['round']))
tr03 = label(TR03)[['raceId', 'year', 'round', 'race', 'driverId', 'driver'] + FINAL_FEATURES + ['podium']]; tr03.to_csv(PROC_DIR / 'training_table.csv', index=False)

out = label(VAL)[['raceId', 'year', 'round', 'race', 'driverId', 'driver', 'grid', 'positionOrder', 'podium']].rename(columns={'grid': 'grid_slot', 'positionOrder': 'finish_pos', 'podium': 'actual_podium'}).copy()
for n, col in [('Baseline (grid only)', 'p_baseline'), ('Logistic regression', 'p_logreg'), ('Gradient boosting', 'p_gboost'), ('Neural network', 'p_nn')]:
    out[col] = P_VA[n]; out[col.replace('p_', 'yes_')] = (P_VA[n] >= BEST_T[n]).astype(int)
out['rank_in_race_nn'] = out.groupby('raceId').p_nn.rank(ascending=False, method='first').astype(int)
out = out.sort_values(['year', 'round', 'rank_in_race_nn']).reset_index(drop=True); out.to_csv(PROC_DIR / 'validation_predictions.csv', index=False)
print(f'training_table.csv: {len(tr03):,} rows x {tr03.shape[1]} columns   |   validation_predictions.csv: {len(out):,} rows x {out.shape[1]} columns')
last = out[out.raceId == out.raceId.iloc[-1]]
print('\nExample — last validation race:', last.race.iloc[0], last.year.iloc[0], '(top 6 by neural-network probability)')
print(last.head(6)[['driver', 'grid_slot', 'p_logreg', 'p_gboost', 'p_nn', 'yes_nn', 'finish_pos', 'actual_podium']].round(2).to_string(index=False))

---
# Part 5 · Experiments

## 11.1 Feature-group ablation
Remove **one group of features at a time**, retrain with the tuned settings, and see how far the validation score falls. A big drop = the group matters; almost none = it adds little. (Baseline excluded — it only uses the grid.)

In [ ]:
GROUPS_FINAL = {'A start position': ['grid_pos', 'quali_pos', 'gap_pct', 'reached_q2', 'reached_q3', 'knockout_format'],
                'B driver form': ['drv_podium_5', 'drv_finish_10', 'drv_points_5', 'drv_dnf_5', 'drv_starts'],
                'C team form': ['con_podium_3', 'con_points_5', 'con_grid_5', 'con_mech_10'],
                'D standings': ['drv_st_points', 'drv_st_position', 'drv_st_wins', 'con_st_points', 'con_st_position', 'con_st_wins'],
                'E circuit': ['drv_circ_podiums']}
assert sorted(sum(GROUPS_FINAL.values(), [])) == sorted(FINAL_FEATURES)
ABL_MODELS = MODELS[1:]

def val_scores(name, feats, seeds):
    ps, rs = [], []
    for sd in seeds:
        m = make_model(name, sd, TUNED.get(name)).fit(TR03[feats], TR03.podium); p = m.predict_proba(VAL[feats])[:, 1]
        ps.append(average_precision_score(VAL.podium, p)); rs.append(roc_auc_score(VAL.podium, p))
    return np.mean(ps), np.mean(rs)

abl = {}
for n in ABL_MODELS:
    seeds = range(3) if n == 'Neural network' else [0]
    abl[(n, 'all features')] = val_scores(n, FINAL_FEATURES, seeds)
    for g_, fs in GROUPS_FINAL.items():
        abl[(n, g_)] = val_scores(n, [f for f in FINAL_FEATURES if f not in fs], seeds)
        abl[(n, 'only ' + g_)] = val_scores(n, fs, seeds)
    abl[(n, 'only A + D')] = val_scores(n, GROUPS_FINAL['A start position'] + GROUPS_FINAL['D standings'], seeds)
A_ = pd.DataFrame(abl, index=['PR_AUC', 'ROC_AUC']).T.rename_axis(['model', 'removed']).reset_index()
piv = A_.pivot(index='removed', columns='model', values='PR_AUC')[ABL_MODELS]; full = piv.loc['all features']
drop = (full - piv.loc[list(GROUPS_FINAL)]).loc[list(GROUPS_FINAL)]
only = piv.loc[['only ' + g_ for g_ in GROUPS_FINAL] + ['only A + D']]
print('PR-AUC on validation with ALL features:', full.round(3).to_dict())
print('\nPR-AUC DROP when a group is removed (bigger = more important; negative = score went up)'); print(drop.round(3).to_string())
print(f'\nPR-AUC when ONLY this group is used (baseline with the grid alone = {average_precision_score(VAL.podium, P_VA["Baseline (grid only)"]):.3f})'); print(only.round(3).to_string())
A_.to_csv(PROC_DIR / 'ablation_results.csv', index=False)

fig, ax = plt.subplots(figsize=(8.5, 3.6)); w = 0.26
for i, (n, c) in enumerate(zip(ABL_MODELS, [BLUE, ORANGE, MAGENTA])): ax.bar(np.arange(len(drop)) + (i - 1) * w, drop[n], w, color=c, label=n, zorder=3)
ax.axhline(0, color=INK2, lw=0.8); ax.set_xticks(range(len(drop))); ax.set_xticklabels(drop.index, fontsize=8.5); ax.set_ylabel('PR-AUC lost when removed'); ax.set_title('What each feature group is worth'); ax.legend(fontsize=8)
save(fig, '20_ablation')

**Reading the ablation**
- **Start position is the only group whose removal clearly hurts** (largest PR-AUC drop in the first table).
- Removing driver form, team form or circuit changes little, within the validation noise. The other groups carry overlapping information (standings and form are strongly correlated).
- **Alone,** each group still has some signal (second table; the grid-only baseline is printed above it, and a random ranking would score about the podium rate).
- The extra row **"only A + D"** (start position + standings) tests the idea that a smaller set would do about as well; compare it with "all features": it scores about as well, so the other groups add little. We keep all groups because they do no harm, and the evidence says the history features add only a little once the start position is known.

## 11.2 Preprocessing order
Does the order of the preparation steps matter? Same tuned logistic regression and neural network, same data; only the preprocessing changes. (Trees do not need scaling, so gradient boosting is left out.)

| Variant | Steps |
|---|---|
| **A (ours)** | split first → fill missing + flag → scale; everything **fitted on training rows only** |
| B | scale first → then fill missing |
| C | fill + scale **fitted on training + validation rows** (information leak) |
| D | fill missing, **no scaling** |

In [ ]:
from sklearn.pipeline import Pipeline
Xtr_raw, Xva_raw = TR03[FINAL_FEATURES], VAL[FINAL_FEATURES]
imp = lambda: SimpleImputer(strategy='median', add_indicator=True)
VARIANTS = {'A  split → fill → scale (ours)': (lambda: make_pipeline(imp(), StandardScaler()), 'train'),
            'B  scale → fill': (lambda: make_pipeline(StandardScaler(), imp()), 'train'),
            'C  fit on train + validation (leak)': (lambda: make_pipeline(imp(), StandardScaler()), 'train+val'),
            'D  fill only, no scaling': (lambda: make_pipeline(imp()), 'train')}
rows = []
for vn, (mk, fit_on) in VARIANTS.items():
    pre = mk().fit(pd.concat([Xtr_raw, Xva_raw]) if fit_on == 'train+val' else Xtr_raw); Xt, Xv = pre.transform(Xtr_raw), pre.transform(Xva_raw)
    lr = LogisticRegression(C=TUNED['Logistic regression']['C'], max_iter=3000).fit(Xt, TR03.podium); p = lr.predict_proba(Xv)[:, 1]
    r = dict(variant=vn, LR_PR=average_precision_score(VAL.podium, p), LR_ROC=roc_auc_score(VAL.podium, p))
    ps, rs = [], []
    for sd in range(3):
        nn_ = TorchMLP(seed=sd, hidden=TUNED['Neural network']['hidden'], dropout=TUNED['Neural network']['dropout']).fit(Xt, TR03.podium); p = nn_.predict_proba(Xv)[:, 1]
        ps.append(average_precision_score(VAL.podium, p)); rs.append(roc_auc_score(VAL.podium, p))
    r.update(NN_PR=np.mean(ps), NN_ROC=np.mean(rs)); rows.append(r)
pre_res = pd.DataFrame(rows).set_index('variant'); print(pre_res.round(4).to_string())
pre_res.to_csv(PROC_DIR / 'preprocessing_order_results.csv')

**Reading the preprocessing test**
- **The order made no practical difference here:** the variants score within a few thousandths of PR-AUC of each other, well inside the noise.
- The leaky variant C is not better — the scaler's averages barely change with the extra validation rows. The rule "fit on training rows only" is about principle (the validation and test years must stay unseen), not about gaining score.
- **Scaling matters for the neural network** (variant D vs A) more than for logistic regression: networks need inputs on similar scales.
- **We keep variant A.**

---
# Part 6 · Final test (2022–2024)

Run **once**. Everything is frozen first: the features, the tuned settings (§10.3), the models trained on 2003–2019, and each model's threshold (§10.4). Nothing below changes any of them. All four models are tested.

In [ ]:
TEST = ALL[ALL.split == 'test'].sort_values('race_idx')
P_TE = {n: proba(n, TEST) for n in MODELS}                    # same fitted models, no refitting
rows = []
for n in MODELS:
    for lab, df, P in [('train', TR03, P_TR), ('validation', VAL, P_VA), ('test', TEST, P_TE)]:
        y, p = df.podium, P[n]; pred = p >= BEST_T[n]
        rows.append(dict(model=n, set=lab, ROC_AUC=roc_auc_score(y, p), PR_AUC=average_precision_score(y, p), F1=f1_score(y, pred), precision=precision_score(y, pred), recall=recall_score(y, pred)))
final = pd.DataFrame(rows); final.to_csv(PROC_DIR / 'metrics_train_validation_test.csv', index=False)
print('Thresholds used:', {n: BEST_T[n] for n in MODELS}); print()
print(final[final.set == 'test'].drop(columns='set').round(3).to_string(index=False))
print('\nF1 by set'); print(final.pivot(index='model', columns='set', values='F1')[['train', 'validation', 'test']].loc[MODELS].round(3).to_string())
print('\nPR-AUC by set'); print(final.pivot(index='model', columns='set', values='PR_AUC')[['train', 'validation', 'test']].loc[MODELS].round(3).to_string())

# how many podium places did each model get right? (expected vs predicted, counted over all test rows)
conf = []
for n in MODELS:
    pred = P_TE[n] >= BEST_T[n]; y = TEST.podium.values.astype(bool)
    conf.append(dict(model=n, real_podiums=int(y.sum()), predicted_yes=int(pred.sum()), correct_yes=int((pred & y).sum()), wrong_yes=int((pred & ~y).sum()), missed_podiums=int((~pred & y).sum())))
print('\nExpected vs predicted on the test rows'); print(pd.DataFrame(conf).to_string(index=False))

# noise on the test years: resample races
rng = np.random.default_rng(1); races = TEST.raceId.unique(); p0 = pd.Series(P_TE['Logistic regression'], index=TEST.index); bs = []
for _ in range(300):
    pick = rng.choice(races, len(races)); idx = np.concatenate([TEST.index[TEST.raceId == r] for r in pick]); bs.append(average_precision_score(TEST.loc[idx, 'podium'], p0.loc[idx]))
print(f'\ntest noise (bootstrap over races): ±{np.std(bs):.3f} PR-AUC')

**Reading the test results**
- **Scores are usually lower than on validation**, as expected: 2022 brought new technical rules, so the test is a harder "new era" check.
- **The models beat the grid-only baseline by a modest margin** (compare the baseline row with the others in the tables above); the baseline's own score barely changed between validation and test, so the gain is not just luck of the period.
- **No ranking of the models is claimed.** The gaps between the models are smaller than the test noise printed above, so the four are reported side by side. (Logistic regression is used for the worked examples in §13 because of the simplicity rule in §10.6, not because of these scores.)
- **Thresholds are too generous for these years.** The "expected vs predicted" table shows the models name more drivers than there are real podium places: they find most podiums, but a sizeable share of the names are wrong. Thresholds chosen on 2020–21 do not transfer perfectly to a more competitive era — a limitation to report.

## 12.2 Expected vs predicted, row by row
One row per driver per race. `expected_podium` is what really happened; `p_*` is the model's probability; `yes_*` is the yes/no answer at the model's threshold. Saved as `test_predictions.csv`.

In [ ]:
tt = label(TEST)[['raceId', 'year', 'round', 'race', 'driverId', 'driver', 'grid', 'positionOrder', 'podium']].rename(columns={'grid': 'grid_slot', 'positionOrder': 'finish_pos', 'podium': 'expected_podium'}).copy()
for n, k in [('Baseline (grid only)', 'baseline'), ('Logistic regression', 'logreg'), ('Gradient boosting', 'gboost'), ('Neural network', 'nn')]:
    tt['p_' + k] = P_TE[n]; tt['yes_' + k] = (P_TE[n] >= BEST_T[n]).astype(int)
tt = tt.sort_values(['year', 'round', 'p_nn'], ascending=[True, True, False]).reset_index(drop=True); tt.to_csv(PROC_DIR / 'test_predictions.csv', index=False)
print(f'test_predictions.csv: {len(tt):,} rows x {tt.shape[1]} columns')
last = tt[tt.raceId == tt.raceId.iloc[-1]]
print('\nExample — last test race:', last.race.iloc[0], last.year.iloc[0], '(top 8 by neural-network probability)')
print(last.head(8)[['driver', 'grid_slot', 'finish_pos', 'expected_podium', 'p_logreg', 'yes_logreg', 'p_gboost', 'yes_gboost', 'p_nn', 'yes_nn']].round(2).to_string(index=False))

In [ ]:
# plain-language summary of the example race, generated from the table above
for k, n in [('logreg', 'Logistic regression'), ('gboost', 'Gradient boosting'), ('nn', 'Neural network')]:
    named = last[last['yes_' + k] == 1]; real = last[last.expected_podium == 1]
    hit, wrong, miss = named[named.expected_podium == 1].driver.tolist(), named[named.expected_podium == 0].driver.tolist(), real[real['yes_' + k] == 0].driver.tolist()
    print(f'{n}: correct {hit} | wrong names {wrong} | missed {miss}')
print('\nA driver who starts far back and then finishes on the podium cannot be foreseen from pre-race information.')

---
# Part 7 · Explainability

We explain the three tuned models (logistic regression, gradient boosting, neural network) on the **validation rows**.
- **Global** (what the model relies on overall): permutation importance and SHAP.
- **Local** (why *this* driver got *this* probability): a SHAP waterfall and LIME, for two test-race drivers.

> **These explanations are not causal.** They describe what the fitted models rely on, not what makes a driver reach the podium. Correlated features (for example team and driver points) share credit in arbitrary ways, so a push up or down must not be read as "this feature causes a podium".

## 13.1 Permutation importance
Shuffle one feature's values (breaking its link to the result) and see how much the validation **PR-AUC falls**. A big fall = the model relies on that feature.

In [ ]:
try:
    import shap, lime
except ImportError:
    import subprocess, sys; subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'shap', 'lime'], check=False); import shap, lime
from lime.lime_tabular import LimeTabularExplainer
from sklearn.inspection import permutation_importance
from matplotlib.patches import Patch
XV, yV = VAL[FINAL_FEATURES], VAL.podium
XAI_MODELS = MODELS[1:]; PI = {}
for n in XAI_MODELS:
    r = permutation_importance(FITTED[n][0], XV, yV, scoring='average_precision', n_repeats=10, random_state=0)
    PI[n] = pd.Series(r.importances_mean, index=FINAL_FEATURES)
PI = pd.DataFrame(PI)

def top_bars(axes, table, title):
    for ax, n in zip(axes, XAI_MODELS):
        s_ = table[n].sort_values().tail(8)
        ax.barh(s_.index, s_.values, color=[GROUP_COLOR[GROUP_OF[f]] for f in s_.index], height=0.7, zorder=3); ax.set_title(n, fontsize=10); ax.tick_params(axis='y', labelsize=8.5)
    axes[0].set_xlabel(title, fontsize=8.5); axes[1].legend(handles=[Patch(color=GROUP_COLOR[k], label=LAB[k]) for k in LAB], fontsize=7, loc='lower right')
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6)); top_bars(axes, PI, 'PR-AUC lost when shuffled'); fig.suptitle('Permutation importance (validation rows, top 8)', y=1.02); save(fig, '21_permutation_importance')

def group_perm(model, cols, reps=10, seed=0):
    rng = np.random.default_rng(seed); base = average_precision_score(yV, model.predict_proba(XV)[:, 1]); d = []
    for _ in range(reps):
        X2 = XV.copy(); X2[cols] = XV[cols].values[rng.permutation(len(XV))]; d.append(base - average_precision_score(yV, model.predict_proba(X2)[:, 1]))
    return np.mean(d)
GP = pd.DataFrame({n: {g_: group_perm(FITTED[n][0], fs) for g_, fs in GROUPS_FINAL.items()} for n in XAI_MODELS})
print('Top 5 features by permutation importance (PR-AUC lost)'); print(PI.apply(lambda c: ', '.join(f'{f} ({v:.3f})' for f, v in c.nlargest(5).items())).to_string())
print('\nWhole groups shuffled together (PR-AUC lost)'); print(GP.round(3).to_string())

**Reading it**
- **Start position dominates in all three models:** the start-position features are at the top, and shuffling the whole group costs by far the most PR-AUC (group table above).
- Driver form and standings come next. Team form and circuit are close to zero — the same as the ablation: their information is covered by the other groups.
- Single-feature numbers understate overlapping features (shuffling `grid_pos` is partly covered by `quali_pos`), so the group table is the cleaner view.

## 13.2 SHAP (global)
SHAP splits each prediction into the **push of every feature** (up or down from the average). Averaging the size of the pushes over many drivers shows which features matter most. Settings: permutation explainer, 100 background rows from the training years, 250 validation rows explained, 440 evaluations per row; the neural network is explained with its first seed only.

In [ ]:
import time; t0 = time.time()
bg = TR03[FINAL_FEATURES].sample(100, random_state=0); Xs = XV.sample(250, random_state=0)
def shap_explainer(n):
    m = FITTED[n][0]; f = lambda X, m=m: m.predict_proba(pd.DataFrame(X, columns=FINAL_FEATURES))[:, 1]
    return shap.Explainer(f, shap.maskers.Independent(bg.values, max_samples=100), algorithm='permutation', feature_names=FINAL_FEATURES, seed=0)
EXPL = {n: shap_explainer(n) for n in XAI_MODELS}
SH = {n: EXPL[n](Xs.values, max_evals=440, silent=True) for n in XAI_MODELS}
SHAP_MEAN = pd.DataFrame({n: pd.Series(np.abs(SH[n].values).mean(0), index=FINAL_FEATURES) for n in XAI_MODELS})
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6)); top_bars(axes, SHAP_MEAN, 'mean |SHAP| (probability points)'); fig.suptitle('SHAP importance (validation rows, top 8)', y=1.02); save(fig, '22_shap_importance')
shap.plots.beeswarm(SH['Logistic regression'], max_display=10, show=False); fig = plt.gcf(); fig.set_size_inches(7.5, 4.6); plt.title('SHAP — logistic regression: each dot is one driver-race'); save(fig, '23_shap_beeswarm_logreg')
print(f'({time.time() - t0:.0f} s)  Top 5 by mean |SHAP|'); print(SHAP_MEAN.apply(lambda c: ', '.join(f'{f} ({v:.3f})' for f, v in c.nlargest(5).items())).to_string())

**Reading it**
- **Same picture as permutation importance:** the start-position features lead in all three models (top features printed above); the models differ in which other features they use next.
- In the dot plot, front-of-grid starts push the probability **up** and back-of-grid starts push it **down**, for the whole range of drivers.
- `reached_q2` can show a negative push: it overlaps with `quali_pos` (§9.7), so the model uses the two together. Read it as an overlap effect, not as "reaching Q2 hurts".
- Typical pushes of the other features are small compared with the start position: the models are mostly saying "where you start".

## 13.3 Local explanations: two test-race drivers (Abu Dhabi 2024)
One driver the models got right (**Norris**: started 1st, won) and one they missed (**Leclerc**: started 19th, finished 3rd). Explained with **SHAP** and **LIME** on the logistic regression, so we can see whether the two methods tell the same story.

In [ ]:
LAST = TEST[TEST.race_idx == TEST.race_idx.max()]; names = LAST.driverId.map(nm)
PICK = {'Lando Norris': LAST[names == 'Lando Norris'].iloc[0], 'Charles Leclerc': LAST[names == 'Charles Leclerc'].iloc[0]}
m_lr = FITTED['Logistic regression'][0]
fill = TR03[FINAL_FEATURES].fillna(TR03[FINAL_FEATURES].median())
lime_ex = LimeTabularExplainer(fill.values, feature_names=FINAL_FEATURES, class_names=['no podium', 'podium'], mode='classification', random_state=0)
pf = lambda X: m_lr.predict_proba(pd.DataFrame(X, columns=FINAL_FEATURES))
for who, row in PICK.items():
    x = row[FINAL_FEATURES].astype(float).values
    e = EXPL['Logistic regression'](x.reshape(1, -1), max_evals=1000, silent=True)[0]
    shap.plots.waterfall(e, max_display=9, show=False); fig = plt.gcf(); fig.set_size_inches(7, 4.2); plt.title(f'SHAP — {who}: probability {pf(x.reshape(1, -1))[0, 1]:.2f}, real result: finished {int(row.positionOrder)}'); save(fig, f'24_shap_waterfall_{who.split()[1].lower()}')
    le = lime_ex.explain_instance(x, pf, num_features=8, num_samples=3000); fig = le.as_pyplot_figure(); fig.set_size_inches(7, 3.6); plt.title(f'LIME — {who}'); plt.tight_layout(); save(fig, f'25_lime_{who.split()[1].lower()}')
    lw = dict(le.as_list()); top = pd.Series(e.values, index=FINAL_FEATURES).reindex(pd.Series(e.values, index=FINAL_FEATURES).abs().nlargest(6).index)
    print(f'\n{who}: grid {int(row.grid)}, finished {int(row.positionOrder)}, model probability {pf(x.reshape(1, -1))[0, 1]:.2f} (average driver: {e.base_values:.2f})')
    print(pd.DataFrame({'value': [x[FINAL_FEATURES.index(f)] for f in top.index], 'SHAP push': top.values}, index=top.index).round(3).to_string())
    print('LIME top features:', '; '.join(f'{k} ({v:+.2f})' for k, v in le.as_list()[:5]))
    lime_top2 = [next(f for f in FINAL_FEATURES if f in k) for k, _ in le.as_list()[:2]]
    print('top-2 features  SHAP:', top.index[:2].tolist(), ' LIME:', lime_top2, ' same two:', set(top.index[:2]) == set(lime_top2))

**Reading it**
- **Norris** (probability far above the average driver's, see the printout): lifted mainly by starting 1st and by his championship position.
- **Leclerc** (very low probability): held down by his poor grid slot and qualifying position. His real finish (3rd) was a recovery from far back, which no pre-race feature can show.
- **SHAP and LIME** are compared by their top two features for each driver (the "same two" line in the printout); beyond the top two the lists differ somewhat, as expected for two different methods.
- One oddity: the team's season points can push a driver *down* while his own points push *up*. The two are strongly correlated, so the logistic regression splits the credit between them; read the pair together.

---
# Part 8 · Inference function

`predict_podium(record)` takes **one driver's pre-race record** (a dict of the features; any feature may be left out) and returns the podium probability and the yes/no answer for each model. Missing values are filled with the **training medians** (logistic regression, neural network); the "was missing" flags exist only for features that had gaps in the training years, and gradient boosting handles gaps natively.

In [ ]:
def predict_podium(record, models=None):
    """record: dict {feature: value}. Unknown keys raise an error; missing features are allowed (treated as missing).
    Returns one row per model: probability, threshold, yes/no. Grid slot: use field_size+1 for a pit-lane start. Give reached_q2 / reached_q3 / knockout_format together (they describe one qualifying format)."""
    unknown = set(record) - set(FINAL_FEATURES)
    if unknown: raise ValueError(f'unknown feature(s): {sorted(unknown)}')
    x = pd.DataFrame([{f: record.get(f, np.nan) for f in FINAL_FEATURES}]).astype(float)
    out = []
    for n in (models or MODELS):
        p = float(np.mean([m.predict_proba(x)[:, 1][0] for m in FITTED[n]]))
        out.append(dict(model=n, features_missing=int(x.isna().sum().sum()), probability=round(p, 3), threshold=BEST_T[n], podium=bool(p >= BEST_T[n])))
    return pd.DataFrame(out)

row = PICK['Lando Norris']; complete = {f: float(row[f]) for f in FINAL_FEATURES if pd.notna(row[f])}
partial = {k: v for k, v in complete.items() if k not in ('gap_pct', 'drv_st_position', 'con_st_position', 'con_mech_10', 'drv_circ_podiums', 'drv_dnf_5')}   # e.g. a season opener / missing lap time
print('1) complete record (Lando Norris, Abu Dhabi 2024 — he won):'); print(predict_podium(complete).to_string(index=False))
print(f'\n2) partially-missing record ({len(complete) - len(partial)} of {len(FINAL_FEATURES)} features removed):'); print(predict_podium(partial).to_string(index=False))
print('\n3) almost empty record (only the grid slot):'); print(predict_podium({'grid_pos': 1}).to_string(index=False))
try: predict_podium({'grid_position': 1})
except ValueError as e: print('\n4) wrong feature name ->', e)

**Reading it**
- **Complete record:** compare the four answers in the first printout.
- **Six features missing:** the function still answers, but the probabilities move (compare with the complete record), because missing values are replaced by training medians. Missing information is not neutral — a limitation to report.
- **Only the grid slot:** the models cannot confirm a podium from one feature; compare the printout.
- A wrong feature name is rejected with a clear message.

---
# Part 9 · Limitations and references

## 15.1 Limitations
- **Qualifying coverage drifts.** No qualifying data before 1994, incomplete data in 1996–2002, and the Q1/Q2/Q3 knockout format exists only from 2006 (handled by the `knockout_format` flag and the 2003 start).
- **Era sensitivity.** Training window and test years sit in different technical eras (2022 rule change). Start years made little difference in §10.2, but the test years are a harder check, and the 2022–24 team ranking is less stable than 2014–21 (§8.3).
- **Thresholds** chosen on 2020–21 are too generous for 2022–24: the models name more drivers than there are podium places.
- **Excluded rows:** non-starters, and pit-lane starters who retired on lap 1 (they look like non-starters in the data); a handful of rows.
- **Small samples:** validation and test are small, so differences between models are within noise and no model is declared best.
- **Missing data is not neutral:** the inference function fills gaps with training medians, which moves the probability (§14).
- **Explanations are not causal** (§13).

## 15.2 References
- Dataset: *Formula 1 World Championship (1950–2020)*, Rohan Rao, Kaggle (Ergast data, extended to 2024 as provided for the project); course primer *Introduction for Formula 1 Beginners*.
- scikit-learn (Pedregosa et al., 2011) · PyTorch (Paszke et al., 2019).
- SHAP: Lundberg & Lee, *A Unified Approach to Interpreting Model Predictions*, NeurIPS 2017.
- LIME: Ribeiro, Singh & Guestrin, *“Why Should I Trust You?”*, KDD 2016.